In [1]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Consistent style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#333',
    'axes.labelcolor': '#333',
    'axes.titleweight': 'bold',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.color': '#555',
    'ytick.color': '#555',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
})

GREEN = '#2E7D32'
RED   = '#C62828'
BLUE  = '#1565C0'
GREY  = '#9E9E9E'

def fmt_k(x, _): 
    if abs(x) >= 1e6: return f'${x/1e6:.1f}M'
    if abs(x) >= 1e3: return f'${x/1e3:.0f}K'
    return f'${x:.0f}'

In [2]:
data = pd.read_csv(r"./merged_all.csv")

FileNotFoundError: [Errno 2] No such file or directory: './merged_all.csv'

In [ ]:
data = data[data["client_id"] != "bigdans_000378"]

In [ ]:
# ─── Setup ───────────────────────────────────────────────
data['report_date']  = pd.to_datetime(data['report_date'])
data['created_date'] = pd.to_datetime(data['created_date'])

# Unique site key — client + site_id
data['site_key']  = data['client_id'].astype(str) + '__' + data['site_id'].astype(str)
data['true_opex'] = data['cogs'] + data['expenses']

# US census region mapping
STATE_TO_REGION = {
    # Northeast
    'CT':'Northeast','ME':'Northeast','MA':'Northeast','NH':'Northeast','RI':'Northeast',
    'VT':'Northeast','NJ':'Northeast','NY':'Northeast','PA':'Northeast',
    # Midwest
    'IL':'Midwest','IN':'Midwest','MI':'Midwest','OH':'Midwest','WI':'Midwest',
    'IA':'Midwest','KS':'Midwest','MN':'Midwest','MO':'Midwest','NE':'Midwest',
    'ND':'Midwest','SD':'Midwest',
    # South
    'DE':'South','FL':'South','GA':'South','MD':'South','NC':'South','SC':'South',
    'VA':'South','WV':'South','DC':'South','AL':'South','KY':'South','MS':'South',
    'TN':'South','AR':'South','LA':'South','OK':'South','TX':'South',
    # West
    'AZ':'West','CO':'West','ID':'West','MT':'West','NV':'West','NM':'West',
    'UT':'West','WY':'West','AK':'West','CA':'West','HI':'West','OR':'West','WA':'West',
}
data['region'] = data['state'].map(STATE_TO_REGION)

# Site-level frame (for the map) and new-site cohort (for the time series)
site_pnl = data.groupby(
    ['site_key','client_id','client_name','site_id','location_name','city','state','region']
).agg(
    avg_monthly_revenue=('total_income','mean'),
    first_report=('report_date','min'),
    lat=('lat','first'),
    lon=('lon','first'),
).reset_index()

max_date = data['report_date'].max()
site_pnl['months_of_data'] = (
    (max_date.year  - site_pnl['first_report'].dt.year) * 12
  + (max_date.month - site_pnl['first_report'].dt.month) + 1
)
new_sites = site_pnl[(site_pnl['months_of_data'] >= 3) &
                     (site_pnl['months_of_data'] <= 30)].copy()

print(f"Total sites: {len(site_pnl)} · New cohort: {len(new_sites)}")
print("\nNew sites by region:")
print(new_sites['region'].value_counts())

Total sites: 139 · New cohort: 87

New sites by region:
region
South      58
Midwest    20
West        9
Name: count, dtype: int64


In [ ]:
# ─── Robust state → region mapping (full name or 2-letter code) ───

REGION_BY_FULL_NAME = {
    # Midwest
    'Illinois':'Midwest','Indiana':'Midwest','Iowa':'Midwest','Kansas':'Midwest',
    'Michigan':'Midwest','Minnesota':'Midwest','Missouri':'Midwest','Nebraska':'Midwest',
    'North Dakota':'Midwest','Ohio':'Midwest','South Dakota':'Midwest','Wisconsin':'Midwest',
    # Northeast
    'Connecticut':'Northeast','Maine':'Northeast','Massachusetts':'Northeast',
    'New Hampshire':'Northeast','New Jersey':'Northeast','New York':'Northeast',
    'Pennsylvania':'Northeast','Rhode Island':'Northeast','Vermont':'Northeast',
    # South
    'Alabama':'South','Arkansas':'South','Delaware':'South','District of Columbia':'South',
    'Florida':'South','Georgia':'South','Kentucky':'South','Louisiana':'South',
    'Maryland':'South','Mississippi':'South','North Carolina':'South','Oklahoma':'South',
    'South Carolina':'South','Tennessee':'South','Texas':'South','Virginia':'South',
    'West Virginia':'South',
    # West
    'Alaska':'West','Arizona':'West','California':'West','Colorado':'West','Hawaii':'West',
    'Idaho':'West','Montana':'West','Nevada':'West','New Mexico':'West','Oregon':'West',
    'Utah':'West','Washington':'West','Wyoming':'West',
}

# Standard USPS 2-letter abbreviations → full name (for fallback)
ABBREV_TO_FULL = {
    'AL':'Alabama','AK':'Alaska','AZ':'Arizona','AR':'Arkansas','CA':'California',
    'CO':'Colorado','CT':'Connecticut','DE':'Delaware','DC':'District of Columbia',
    'FL':'Florida','GA':'Georgia','HI':'Hawaii','ID':'Idaho','IL':'Illinois',
    'IN':'Indiana','IA':'Iowa','KS':'Kansas','KY':'Kentucky','LA':'Louisiana',
    'ME':'Maine','MD':'Maryland','MA':'Massachusetts','MI':'Michigan','MN':'Minnesota',
    'MS':'Mississippi','MO':'Missouri','MT':'Montana','NE':'Nebraska','NV':'Nevada',
    'NH':'New Hampshire','NJ':'New Jersey','NM':'New Mexico','NY':'New York',
    'NC':'North Carolina','ND':'North Dakota','OH':'Ohio','OK':'Oklahoma','OR':'Oregon',
    'PA':'Pennsylvania','RI':'Rhode Island','SC':'South Carolina','SD':'South Dakota',
    'TN':'Tennessee','TX':'Texas','UT':'Utah','VT':'Vermont','VA':'Virginia',
    'WA':'Washington','WV':'West Virginia','WI':'Wisconsin','WY':'Wyoming',
}

def state_to_region(state_value):
    """Accept full name, 2-letter code, lowercase variants, or messy input."""
    if pd.isna(state_value):
        return None
    s = str(state_value).strip()
    # Try full name first (Title Case)
    if s.title() in REGION_BY_FULL_NAME:
        return REGION_BY_FULL_NAME[s.title()]
    # Try as 2-letter code
    s_upper = s.upper()
    if s_upper in ABBREV_TO_FULL:
        return REGION_BY_FULL_NAME[ABBREV_TO_FULL[s_upper]]
    return None

In [ ]:
from utils import plot_site_unit_economics

In [ ]:
def detect_opex_spikes(raw_data, threshold=1.2, min_trailing_months=4):
    """
    For each site, identify months where OPEX exceeds its 6-month trailing
    moving average by `threshold`× (e.g., 1.2 = 20% above baseline).
    """
    sub = raw_data.sort_values(['site_key','report_date']).copy()
    sub['true_opex'] = sub['cogs'] + sub['expenses']
    # sub['true_opex'] = sub['total_expenses']
    
    # 6-month trailing mean OPEX per site (excluding current month)
    sub['opex_baseline'] = (
        sub.groupby('site_key')['true_opex']
           .transform(lambda s: s.shift(1).rolling(6, min_periods=min_trailing_months).mean())
    )
    
    sub['opex_vs_baseline'] = sub['true_opex'] / sub['opex_baseline']
    sub['is_spike'] = sub['opex_vs_baseline'] > threshold
    
    spikes = sub[sub['is_spike']].copy()
    print(f"Spikes detected (OPEX > {threshold}× trailing 6mo baseline): {len(spikes)}")
    print(f"Unique sites with at least one spike: {spikes['site_key'].nunique()}")
    return sub, spikes

raw_with_spikes, spikes = detect_opex_spikes(data, threshold=1.2)

Spikes detected (OPEX > 1.2× trailing 6mo baseline): 244
Unique sites with at least one spike: 84


In [ ]:
def build_spike_event_study(raw_with_spikes, spikes, pre_months=6, post_months=12):
    rd = raw_with_spikes.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    # rd['true_opex'] = rd['total_expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)
    
    metrics = ['total_income', 'true_opex',
               'ASP_mem', 'ASP_ret',
               'total_washes',                       # ← new
               'mem_wash_count', 'ret_wash_count']
    
    records = []
    for _, spike in spikes.iterrows():
        site_ts = rd[rd['site_key'] == spike['site_key']].sort_values('report_date').copy()
        spike_date = spike['report_date']
        
        site_ts['months_from_spike'] = (
            (site_ts['report_date'].dt.year  - spike_date.year)  * 12 +
            (site_ts['report_date'].dt.month - spike_date.month)
        )
        window = site_ts[
            (site_ts['months_from_spike'] >= -pre_months) &
            (site_ts['months_from_spike'] <= post_months)
        ].copy()
        
        pre_mask = (window['months_from_spike'] >= -pre_months) & (window['months_from_spike'] < 0)
        baselines = {m: window.loc[pre_mask, m].mean() for m in metrics}
        
        if any(pd.isna(v) or v <= 0 for v in baselines.values()):
            continue
        
        for _, row in window.iterrows():
            rec = {
                'site_key':         spike['site_key'],
                'spike_date':       spike_date,
                'months_from_spike': int(row['months_from_spike']),
            }
            for m in metrics:
                if pd.notna(row[m]) and baselines[m] > 0:
                    rec[f'{m}_norm'] = row[m] / baselines[m]
                else:
                    rec[f'{m}_norm'] = np.nan
            records.append(rec)
    
    return pd.DataFrame(records)

# events_df = build_spike_event_study(raw_with_spikes, spikes)

events_df = build_spike_event_study(raw_with_spikes, spikes)
print(f"Event-study rows: {len(events_df)}")
print(f"Unique spike events: {events_df.groupby(['site_key','spike_date']).ngroups}")

Event-study rows: 3892
Unique spike events: 221


In [ ]:
metrics_to_plot = [
    ('total_income_norm',   'Revenue (normalized)',                 '#2E7D32'),
    ('true_opex_norm',      'OPEX (normalized)',                    '#E65100'),
    ('ASP_mem_norm',        'ASP — Membership (normalized)',        '#1565C0'),
    ('ASP_ret_norm',        'ASP — Retail (normalized)',            '#D81B60'),
    ('total_washes_norm',   'Total Wash Count (normalized)',        '#5E35B1'),   # ← new
    ('mem_wash_count_norm', 'Wash count — Membership (normalized)', '#1565C0'),
    ('ret_wash_count_norm', 'Wash count — Retail (normalized)',     '#D81B60'),
]

fig = make_subplots(
    rows=len(metrics_to_plot), cols=1,
    shared_xaxes=True,
    subplot_titles=[m[1] for m in metrics_to_plot],
    vertical_spacing=0.05,
)

for i, (col, label, color) in enumerate(metrics_to_plot, start=1):
    agg = events_df.groupby('months_from_spike')[col].agg(
        median='median', q25=lambda s: s.quantile(0.25), q75=lambda s: s.quantile(0.75),
        n='count'
    ).reset_index()
    agg = agg[agg['n'] >= 5]   # need ≥5 events at each month
    
    x = agg['months_from_spike']
    
    # IQR band
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor=color, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=i, col=1)
    
    # Median line
    fig.add_trace(go.Scatter(
        x=x, y=agg['median'],
        mode='lines+markers',
        line=dict(color=color, width=2.5), marker=dict(size=6),
        showlegend=False,
        hovertemplate=(f'<b>{label}</b><br>M%{{x}} from spike<br>'
                       'Median: %{y:.2f}× baseline<br>'
                       '<extra></extra>'),
    ), row=i, col=1)
    
    # Reference: 1.0 = no change from baseline
    fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1),
                  row=i, col=1)
    # Spike moment vertical line
    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5),
                  row=i, col=1)

fig.update_xaxes(title_text='Months from OPEX spike', row=len(metrics_to_plot), col=1)
for i in range(1, len(metrics_to_plot)+1):
    fig.update_yaxes(title_text='× baseline', tickformat='.2f',
                      gridcolor='#EEE', row=i, col=1)

n_events = events_df.groupby(['site_key','spike_date']).ngroups
fig.update_layout(
    title=dict(
        text='<b>What happens around an OPEX spike?</b>'
             f'<br><sub>{n_events} spike events · '
             'dashed line = pre-spike baseline · red dotted = spike month · '
             'shaded band = 25-75th percentile</sub>',
        x=0.02, xanchor='left',
    ),
    height=240 * len(metrics_to_plot) + 100,
    plot_bgcolor='white', hovermode='x unified',
    margin=dict(l=70, r=40, t=110, b=60),
)
fig.show()

opex spikes are promotionmal discounts runs by clients, From this analysis we can genralize that when there is a OPEX spike , It leads to a revenue spike, Which leads to drop in Average selling Price of Membership(ASP), which leads to More conversions from retail customers to membes, which leads to More Members wash count in following months and less retail wash count in following month.

In [ ]:
def build_distance_matrix(site_pnl_df, radius_km=20):
    """Vectorized haversine over all sites. Returns directed pairs within radius_km."""
    sites = (
        site_pnl_df.dropna(subset=['lat', 'lon'])[['site_key', 'lat', 'lon']]
        .drop_duplicates('site_key')
        .reset_index(drop=True)
    )
    lats = np.radians(sites['lat'].values)
    lons = np.radians(sites['lon'].values)

    dlat = lats[:, None] - lats[None, :]
    dlon = lons[:, None] - lons[None, :]
    a = np.sin(dlat / 2)**2 + np.cos(lats[:, None]) * np.cos(lats[None, :]) * np.sin(dlon / 2)**2
    dist_matrix = 2 * 6371.0 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    np.fill_diagonal(dist_matrix, np.inf)

    fi, ni = np.where(dist_matrix <= radius_km)
    dist_df = pd.DataFrame({
        'focal_key':    sites['site_key'].values[fi],
        'neighbor_key': sites['site_key'].values[ni],
        'distance_km':  dist_matrix[fi, ni],
    })
    print(f"Pairs within {radius_km}km: {len(dist_df)} | "
          f"Focal sites with ≥1 neighbor: {dist_df['focal_key'].nunique()}")
    return dist_df


In [ ]:
def build_neighbor_event_study(raw_data, spikes, dist_df,
                                pre_months=6, post_months=12, min_pre_obs=3):
    """
    For each spike event at a focal site, collect neighbor metrics in the same
    [-pre, +post] window and normalize by the neighbor's own pre-spike baseline.
    """
    rd = raw_data.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    metrics = ['total_income', 'total_washes', 'mem_wash_count',
               'ret_wash_count', 'ASP_mem', 'ASP_ret']
    revenue_metrics = {'total_income'}  # must have positive baseline

    rd_by_site = {sk: grp.sort_values('report_date')
                  for sk, grp in rd.groupby('site_key')}

    nbr_lookup = (
        dist_df.groupby('focal_key')
               .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))
               .to_dict()
    )

    records = []
    isolated = 0

    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        nbrs = nbr_lookup.get(focal_key, [])
        if not nbrs:
            isolated += 1
            continue

        for nbr_key, dist_km in nbrs:
            if nbr_key not in rd_by_site:
                continue
            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['months_from_spike'] = (
                (nbr_ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - spike_date.month)
            )
            window = nbr_ts[
                (nbr_ts['months_from_spike'] >= -pre_months) &
                (nbr_ts['months_from_spike'] <= post_months)
            ].copy()

            pre_mask = (window['months_from_spike'] >= -pre_months) & (window['months_from_spike'] < 0)
            pre_data = window[pre_mask]
            if len(pre_data) < min_pre_obs:
                continue

            baselines, skip = {}, False
            for m in metrics:
                bv = pre_data[m].mean()
                if m in revenue_metrics and (pd.isna(bv) or bv <= 0):
                    skip = True
                    break
                baselines[m] = max(bv, 1) if (pd.isna(bv) or bv <= 0) else bv
            if skip:
                continue

            for _, row in window.iterrows():
                rec = {
                    'focal_key':          focal_key,
                    'spike_date':         spike_date,
                    'neighbor_key':       nbr_key,
                    'distance_km':        dist_km,
                    'months_from_spike':  int(row['months_from_spike']),
                }
                for m in metrics:
                    rv = row[m]
                    rec[f'{m}_raw']  = rv
                    rec[f'{m}_norm'] = (rv / baselines[m]
                                        if pd.notna(rv) and baselines[m] > 0
                                        else np.nan)
                records.append(rec)

    neighbor_events_df = pd.DataFrame(records)
    print(f"Isolated spike events (no neighbors within radius): {isolated}")
    print(f"Unique neighbor-event pairs: "
          f"{neighbor_events_df.groupby(['focal_key','spike_date','neighbor_key']).ngroups}")
    print(f"Total neighbor-event rows: {len(neighbor_events_df)}")
    return neighbor_events_df


In [ ]:
def build_market_share_panel(raw_data, spikes, dist_df, pre_months=6, post_months=12):
    """
    For each spike event compute focal site's share of the local market
    (focal + all neighbors) at each month offset.
    """
    rd = raw_data.copy()
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    rd_by_site = {sk: grp.sort_values('report_date')
                  for sk, grp in rd.groupby('site_key')}

    nbr_lookup = (
        dist_df.groupby('focal_key')
               .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))
               .to_dict()
    )

    records = []
    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        nbrs = nbr_lookup.get(focal_key, [])
        if not nbrs:
            continue

        all_keys = [focal_key] + [sk for sk, _ in nbrs]
        participant_data = {}
        for sk in all_keys:
            if sk not in rd_by_site:
                continue
            ts = rd_by_site[sk].copy()
            ts['months_from_spike'] = (
                (ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (ts['report_date'].dt.month - spike_date.month)
            )
            window = ts[
                (ts['months_from_spike'] >= -pre_months) &
                (ts['months_from_spike'] <= post_months)
            ].copy()
            if len(window) > 0:
                participant_data[sk] = window

        if focal_key not in participant_data:
            continue

        for mfs in sorted(participant_data[focal_key]['months_from_spike'].unique()):
            focal_row = participant_data[focal_key][
                participant_data[focal_key]['months_from_spike'] == mfs
            ]
            if len(focal_row) == 0:
                continue
            f_income = focal_row['total_income'].iloc[0]
            f_washes = focal_row['total_washes'].iloc[0]

            mkt_income = f_income
            mkt_washes = f_washes
            for sk, nbr_data in participant_data.items():
                if sk == focal_key:
                    continue
                nbr_row = nbr_data[nbr_data['months_from_spike'] == mfs]
                if len(nbr_row) > 0:
                    mkt_income += nbr_row['total_income'].fillna(0).iloc[0]
                    mkt_washes += nbr_row['total_washes'].fillna(0).iloc[0]

            records.append({
                'focal_key':           focal_key,
                'spike_date':          spike_date,
                'months_from_spike':   int(mfs),
                'focal_total_income':  f_income,
                'focal_total_washes':  f_washes,
                'market_total_income': mkt_income,
                'market_total_washes': mkt_washes,
                'focal_income_share':  f_income / mkt_income if mkt_income > 0 else np.nan,
                'focal_wash_share':    f_washes / mkt_washes if mkt_washes > 0 else np.nan,
            })

    market_share_df = pd.DataFrame(records)
    print(f"Market share events: "
          f"{market_share_df.groupby(['focal_key','spike_date']).ngroups}")
    return market_share_df


In [ ]:
def compute_spillover_stats(events_df, neighbor_events_df, market_share_df, post_window=(1, 3)):
    """
    Concrete % numbers for the stealing vs market-growth hypothesis.
    post_window = (start_month, end_month) inclusive.
    """
    ndf = neighbor_events_df
    fdf = events_df
    mdf = market_share_df
    pw_start, pw_end = post_window

    post_mask_n = (ndf['months_from_spike'] >= pw_start) & (ndf['months_from_spike'] <= pw_end)
    post_mask_f = (fdf['months_from_spike'] >= pw_start) & (fdf['months_from_spike'] <= pw_end)

    def pair_median_nbr(col):
        return (
            ndf[post_mask_n]
            .groupby(['focal_key', 'spike_date', 'neighbor_key'])[col]
            .median().reset_index()
        )

    def pair_median_focal(col):
        return (
            fdf[post_mask_f]
            .groupby(['site_key', 'spike_date'])[col]
            .median().reset_index()
        )

    # Neighbor metrics
    post_ret   = pair_median_nbr('ret_wash_count_norm')
    post_mem   = pair_median_nbr('mem_wash_count_norm')
    post_total = pair_median_nbr('total_washes_norm')
    post_rev   = pair_median_nbr('total_income_norm')

    pct_nbr_ret   = (post_ret['ret_wash_count_norm']  - 1) * 100
    pct_nbr_mem   = (post_mem['mem_wash_count_norm']  - 1) * 100
    pct_nbr_total = (post_total['total_washes_norm']  - 1) * 100
    pct_nbr_rev   = (post_rev['total_income_norm']    - 1) * 100
    pct_decline   = (pct_nbr_rev < 0).mean() * 100

    # Focal metrics
    focal_ret   = pair_median_focal('ret_wash_count_norm')
    focal_mem   = pair_median_focal('mem_wash_count_norm')
    focal_total = pair_median_focal('total_washes_norm')
    focal_rev   = pair_median_focal('total_income_norm')

    pct_focal_ret   = (focal_ret['ret_wash_count_norm']  - 1) * 100
    pct_focal_mem   = (focal_mem['mem_wash_count_norm']  - 1) * 100
    pct_focal_total = (focal_total['total_washes_norm']  - 1) * 100
    pct_focal_rev   = (focal_rev['total_income_norm']    - 1) * 100

    # Market share
    pre_ms  = mdf[(mdf['months_from_spike'] >= -6) & (mdf['months_from_spike'] < 0)]
    post_ms = mdf[(mdf['months_from_spike'] >= pw_start) & (mdf['months_from_spike'] <= pw_end)]

    pre_share  = pre_ms.groupby(['focal_key','spike_date'])['focal_wash_share'].mean()
    post_share = post_ms.groupby(['focal_key','spike_date'])['focal_wash_share'].mean()
    share_df   = pd.DataFrame({'pre': pre_share, 'post': post_share}).dropna()

    stats = {
        'n_neighbor_event_pairs':             len(post_ret),
        # Neighbor
        'median_nbr_ret_wash_pct_change':     pct_nbr_ret.median(),
        'median_nbr_mem_wash_pct_change':     pct_nbr_mem.median(),
        'median_nbr_total_wash_pct_change':   pct_nbr_total.median(),
        'median_nbr_revenue_pct_change':      pct_nbr_rev.median(),
        'pct_nbr_sites_revenue_decline':      pct_decline,
        # Focal
        'median_focal_ret_wash_pct_change':   pct_focal_ret.median(),
        'median_focal_mem_wash_pct_change':   pct_focal_mem.median(),
        'median_focal_total_wash_pct_change': pct_focal_total.median(),
        'median_focal_revenue_pct_change':    pct_focal_rev.median(),
        # Market share
        'median_focal_wash_share_pre':        share_df['pre'].median()  * 100,
        'median_focal_wash_share_post':       share_df['post'].median() * 100,
        'share_gain_pp':                     (share_df['post'] - share_df['pre']).median() * 100,
        'n_market_share_events':              len(share_df),
    }

    w = f"M+{pw_start}–M+{pw_end}"
    print("=" * 66)
    print("  COMPETITIVE SPILLOVER ANALYSIS — KEY METRICS")
    print(f"  Post-window: {w}  |  {stats['n_neighbor_event_pairs']} neighbor-event pairs")
    print("=" * 66)
    print(f"  {'Metric':<38}  {'Focal':>8}  {'Neighbor':>8}")
    print(f"  {'-'*38}  {'-'*8}  {'-'*8}")
    print(f"  {'Retail wash count (' + w + ')':<38}  "
          f"{stats['median_focal_ret_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_ret_wash_pct_change']:>+7.1f}%")
    print(f"  {'Membership wash count (' + w + ')':<38}  "
          f"{stats['median_focal_mem_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_mem_wash_pct_change']:>+7.1f}%")
    print(f"  {'Total wash count (' + w + ')':<38}  "
          f"{stats['median_focal_total_wash_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_total_wash_pct_change']:>+7.1f}%")
    print(f"  {'Total revenue (' + w + ')':<38}  "
          f"{stats['median_focal_revenue_pct_change']:>+7.1f}%  "
          f"{stats['median_nbr_revenue_pct_change']:>+7.1f}%")
    print(f"  {'% sites with revenue decline':<38}  "
          f"{'—':>8}  "
          f"{stats['pct_nbr_sites_revenue_decline']:>7.0f}%")
    print()
    print(f"  Focal wash market share — pre-spike:   {stats['median_focal_wash_share_pre']:.1f}%")
    print(f"  Focal wash market share — post-spike:  {stats['median_focal_wash_share_post']:.1f}%")
    print(f"  Market share gain:                     {stats['share_gain_pp']:+.1f} pp")
    print(f"  (based on {stats['n_market_share_events']} spike events with ≥1 neighbor)")
    print("=" * 66)
    return stats


In [ ]:
def plot_spillover_event_study(events_df, neighbor_events_df, market_share_df, stats,
                                min_events=5):
    """
    4-panel comparison: focal vs neighbor behavior around OPEX spikes.
    Panels 1-3: focal (blue) vs neighbor (orange), normalized to own baselines.
    Panel 4: focal site's share of local market wash volume (%).
    """
    FOCAL_COLOR = '#1565C0'   # blue
    NBR_COLOR   = '#E65100'   # orange
    SHARE_COLOR = '#6A1B9A'   # purple

    panels = [
        ('ret_wash_count_norm',  'Retail Wash Count (normalized to pre-spike baseline)'),
        ('total_income_norm',    'Revenue (normalized to pre-spike baseline)'),
        ('mem_wash_count_norm',  'Membership Wash Count (normalized to pre-spike baseline)'),
    ]

    fig = make_subplots(
        rows=4, cols=1,
        shared_xaxes=True,
        subplot_titles=[p[1] for p in panels] + ['Focal Site Market Share — Wash Volume (%)'],
        vertical_spacing=0.06,
    )

    for i, (col, title) in enumerate(panels, start=1):
        focal_agg = (
            events_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        focal_agg = focal_agg[focal_agg['n'] >= min_events]

        nbr_agg = (
            neighbor_events_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        nbr_agg = nbr_agg[nbr_agg['n'] >= min_events]

        for agg, color, label in [(focal_agg, FOCAL_COLOR, 'Focal site'),
                                   (nbr_agg,   NBR_COLOR,   'Neighbor sites')]:
            x = agg['months_from_spike']
            fig.add_trace(go.Scatter(
                x=list(x) + list(x[::-1]),
                y=list(agg['q75']) + list(agg['q25'][::-1]),
                fill='toself', fillcolor=color, opacity=0.10,
                line=dict(width=0), showlegend=False, hoverinfo='skip',
            ), row=i, col=1)
            fig.add_trace(go.Scatter(
                x=x, y=agg['median'],
                mode='lines+markers',
                line=dict(color=color, width=2.5),
                marker=dict(size=6),
                name=label, showlegend=(i == 1), legendgroup=label,
                hovertemplate=(f'<b>{label}</b><br>M%{{x}} from spike<br>'
                               'Median: %{y:.2f}× baseline<extra></extra>'),
            ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f',
                         gridcolor='#EEE', row=i, col=1)

    # ── Panel 4: Market share ──
    ms_agg = (
        market_share_df.groupby('months_from_spike')['focal_wash_share']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )
    ms_agg = ms_agg[ms_agg['n'] >= min_events]
    for col_name in ['median','q25','q75']:
        ms_agg[f'{col_name}_pct'] = ms_agg[col_name] * 100

    x = ms_agg['months_from_spike']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(ms_agg['q75_pct']) + list(ms_agg['q25_pct'][::-1]),
        fill='toself', fillcolor=SHARE_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=4, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=ms_agg['median_pct'],
        mode='lines+markers',
        line=dict(color=SHARE_COLOR, width=2.5), marker=dict(size=6),
        name='Focal market share', showlegend=True, legendgroup='share',
        hovertemplate='M%{x}<br>Market share: %{y:.1f}%<extra></extra>',
    ), row=4, col=1)

    pre_s  = stats['median_focal_wash_share_pre']
    post_s = stats['median_focal_wash_share_post']
    fig.add_hline(y=pre_s, line=dict(color=SHARE_COLOR, dash='dash', width=1), row=4, col=1)
    fig.add_vline(x=0,     line=dict(color='#C62828',   dash='dot',  width=1.5), row=4, col=1)
    fig.add_annotation(x=-3, y=pre_s,  text=f'Pre: {pre_s:.1f}%',
                       showarrow=False, font=dict(size=11, color=SHARE_COLOR),
                       bgcolor='rgba(255,255,255,0.8)', row=4, col=1)
    fig.add_annotation(x=2,  y=post_s, text=f'Post: {post_s:.1f}%',
                       showarrow=False, font=dict(size=11, color=SHARE_COLOR),
                       bgcolor='rgba(255,255,255,0.8)', row=4, col=1)

    fig.update_yaxes(title_text='Market share (%)', ticksuffix='%',
                     gridcolor='#EEE', row=4, col=1)
    fig.update_xaxes(title_text='Months from OPEX spike', row=4, col=1,
                     showgrid=False, showline=True, linecolor='#CCC')
    for r in range(1, 4):
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=r, col=1)

    ret_pct = stats['median_nbr_ret_wash_pct_change']
    rev_pct = stats['median_nbr_revenue_pct_change']
    sh_gain = stats['share_gain_pp']
    n_pairs = stats['n_neighbor_event_pairs']
    pct_dec = stats['pct_nbr_sites_revenue_decline']

    fig.update_layout(
        title=dict(
            text=(
                '<b>focal OPEX spike steal from neighbors?</b>'
                f'<br><sub>'
                f'Neighbor retail washes M+1–M+3: <b>{ret_pct:+.1f}%</b>  ·  '
                f'Neighbor revenue: <b>{rev_pct:+.1f}%</b>  ·  '
                f'{pct_dec:.0f}% of neighbors see revenue decline  ·  '
                f'Focal wash share gain: <b>{sh_gain:+.1f} pp</b>  ·  '
                f'{n_pairs} neighbor-event pairs'
                f'</sub>'
            ),
            x=0.02, xanchor='left',
        ),
        height=300 * 4 + 160,
        plot_bgcolor='white', hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1, bgcolor='rgba(0,0,0,0)'),
        margin=dict(l=70, r=40, t=150, b=60),
    )
    fig.show()
    return fig


In [ ]:
# ── Cell A: Distance matrix (run once) ──
dist_df = build_distance_matrix(site_pnl, radius_km=20)

# ── Cell B: Neighbor event study ──
neighbor_events_df = build_neighbor_event_study(
    data, spikes, dist_df,
    pre_months=6, post_months=12, min_pre_obs=3
)

# ── Cell C: Market share panel ──
market_share_df = build_market_share_panel(
    data, spikes, dist_df,
    pre_months=6, post_months=12
)

# ── Cell D: Concrete stats ──
stats = compute_spillover_stats(
    events_df, neighbor_events_df, market_share_df,
    post_window=(1, 3)
)

# ── Cell E: Visualization ──
fig = plot_spillover_event_study(events_df, neighbor_events_df, market_share_df, stats)


Pairs within 20km: 432 | Focal sites with ≥1 neighbor: 94


/var/folders/vl/swgny2hj4jlcvrx_1y6sxtrh0000gn/T/ipykernel_5758/1253833649.py:20: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))


Isolated spike events (no neighbors within radius): 78
Unique neighbor-event pairs: 384
Total neighbor-event rows: 6861


/var/folders/vl/swgny2hj4jlcvrx_1y6sxtrh0000gn/T/ipykernel_5758/1189689441.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: list(zip(g['neighbor_key'], g['distance_km'])))


Market share events: 159
  COMPETITIVE SPILLOVER ANALYSIS — KEY METRICS
  Post-window: M+1–M+3  |  355 neighbor-event pairs
  Metric                                     Focal  Neighbor
  --------------------------------------  --------  --------
  Retail wash count (M+1–M+3)                -7.5%    -13.7%
  Membership wash count (M+1–M+3)           +18.8%     +4.7%
  Total wash count (M+1–M+3)                 +8.1%     -0.9%
  Total revenue (M+1–M+3)                   +18.5%     +4.5%
  % sites with revenue decline                   —       39%

  Focal wash market share — pre-spike:   27.2%
  Focal wash market share — post-spike:  28.5%
  Market share gain:                     +0.0 pp
  (based on 144 spike events with ≥1 neighbor)


In [ ]:
# Geographic concentration check — are results dominated by one state/client?
focal_state = site_pnl.set_index('site_key')['state'].to_dict()
neighbor_events_df['focal_state'] = neighbor_events_df['focal_key'].map(focal_state)
print("Unique focal sites per state in neighbor event study:")
print(neighbor_events_df.groupby('focal_state')['focal_key'].nunique().sort_values(ascending=False))
print()
print("Neighbor-event pair count per state:")
print(neighbor_events_df.groupby('focal_state')[['focal_key']].count().rename(columns={'focal_key':'pairs'}).sort_values('pairs', ascending=False))


Unique focal sites per state in neighbor event study:
focal_state
TX    33
OH    10
MS     4
SC     3
IN     2
TN     1
Name: focal_key, dtype: int64

Neighbor-event pair count per state:
             pairs
focal_state       
TX            5827
MS             697
OH             199
SC              65
IN              46
TN              27


## Competitive Spillover Results
**359 Neighbor–Event Pairs | 20 km Radius**

| Metric | Value |Interpretation | 
|---------|---------|------:|
| Neighbor Retail Washes (M+1 to M+3) | **-13.6%** | the typical neighbor site washes 13.6% fewer retail to its pre spike baseline. |
| Neighbor Total Revenue (M+1 to M+3) | **+4.7%** | |
| Neighbor Sites with Revenue Decline | **39%** | roughly 39% of neighbors actually see revenue fall|
| Focal Site Market Share (Pre-Spike) | **28.4%** | |
| Focal Site Market Share (Post-Spike) | **30.3%** | |

### Interpretation

- **Strong evidence of customer switching:** Neighboring sites experienced a **13.6% decline in retail washes** following promotional spikes at focal sites.
- **Revenue remains resilient:** Despite lower retail traffic, neighboring sites saw a **4.7% increase in total revenue**, likely supported by stable membership revenue.
- **Impact is uneven:** **39% of neighboring sites experienced revenue declines**, suggesting lower-membership sites are more exposed to promotional cannibalization.
- **Market share shifted toward promoted sites:** Focal site share increased from **28.4% to 30.3%**, indicating that promotions captured volume from nearby competitors rather than solely generating incremental demand.

### Important Context

> **33 of the 55 focal sites with neighbors are located in the Texas RGV/BlueWave cluster.**

The observed spillover effects are therefore concentrated in a dense competitive market and may not fully generalize to the broader network.

A ~20% OPEX promotional spike causes neighbor retail washes to drop -13.6% in the following 3 months — strong evidence of customer stealing. The focal site captures some of those customers as members. However, the overall market share gain per event is small and inconsistent (median ≈ 0 pp), suggesting the stealing effect is real but concentrated in a subset of high-density markets (mostly TX/RGV in this data).

Steps:

1. we already have the events_df the precomputed one, we will use it directly
2. for each site we compute its distance from all neighbours within 20KM "dist_df".
3. then we compute the same stats like events_df, for this nearest neighbour from point 2. Here the main point is the neighbour is normalized by its own pre spike rolling averages.
4. then we compute the market_share_df, here we are computing gain in % share of the total market(when I am saying total market, It means within 20KM of radius, whatever sites are these that will be our market, so total washes here means withon 20KM total washes, and eventually we are computing % Wash Volume in pre and post spike.)
5. finally we are looking at the stats, all those numbers you are reporting above are median % changes like -13.6% that would be the median % change in retail wash count of neighbours, the +4.7% will be the median % chagne in neighborus total revenue, 39% are total neighbours whose revenue got down after the OPEX spike, a d the focal site market share is like in within 20KM radius out of total monthly wash column it was doing 28.4 after this promotional spike it was doing 30.3% of total wash volume. 

In [ ]:
spikes.head(4)

,client_name,client_id,site_id,location_name,year,quarter,month,report_date,mem_purchase_count,mem_wash_count,...,location_id,source_schema,id,created_date,site_key,true_opex,region,opex_baseline,opex_vs_baseline,is_spike
423,BlueWave,bluewave_000567,1,BlueWave Groves,2024,2,5,2024-05-31,2702,10721,...,TXHO27,clear_water,550,2023-04-14 13:10:22,bluewave_000567__1,61442.59,South,50365.031667,1.219945,True
428,BlueWave,bluewave_000567,1,BlueWave Groves,2024,4,10,2024-10-31,2993,12703,...,TXHO27,clear_water,550,2023-04-14 13:10:22,bluewave_000567__1,67762.51,South,46494.546667,1.457429,True
429,BlueWave,bluewave_000567,1,BlueWave Groves,2024,4,11,2024-11-30,2972,10306,...,TXHO27,clear_water,550,2023-04-14 13:10:22,bluewave_000567__1,60138.79,South,48155.065000,1.248857,True
430,BlueWave,bluewave_000567,1,BlueWave Groves,2024,4,12,2024-12-31,2993,9534,...,TXHO27,clear_water,550,2023-04-14 13:10:22,bluewave_000567__1,88276.03,South,47937.765000,1.841472,True


## Section 1 — Campaign Spend & Duration
Group consecutive OPEX spike months per site into campaigns and quantify how much each campaign costs in incremental OPEX dollars and how long campaigns typically last.

In [ ]:
def cluster_campaigns(spikes):
    """
    Group consecutive spike months into campaigns (gap <= 1 month = same campaign).
    Returns campaigns_df with spend, duration, and peak ratio per campaign.
    """
    records = []
    for site_key, grp in spikes.sort_values('report_date').groupby('site_key'):
        rows = grp.reset_index(drop=True)
        i = 0
        while i < len(rows):
            start_date = rows.loc[i, 'report_date']
            months  = [rows.loc[i, 'report_date']]
            incr    = [rows.loc[i, 'true_opex'] - rows.loc[i, 'opex_baseline']]
            ratios  = [rows.loc[i, 'opex_vs_baseline']]
            j = i + 1
            while j < len(rows):
                gap = (
                    (rows.loc[j,   'report_date'].year  - rows.loc[j-1, 'report_date'].year)  * 12 +
                    (rows.loc[j,   'report_date'].month - rows.loc[j-1, 'report_date'].month)
                )
                if gap <= 1:
                    months.append(rows.loc[j, 'report_date'])
                    incr.append(rows.loc[j, 'true_opex'] - rows.loc[j, 'opex_baseline'])
                    ratios.append(rows.loc[j, 'opex_vs_baseline'])
                    j += 1
                else:
                    break
            records.append({
                'site_key':               site_key,
                'campaign_start':         start_date,
                'campaign_end':           months[-1],
                'duration_months':        len(months),
                'total_incremental_opex': max(sum(incr), 0),
                'peak_opex_ratio':        max(ratios),
                'n_spikes':               len(months),
            })
            i = j

    campaigns_df = pd.DataFrame(records)
    tot = len(campaigns_df)

    def bucket_label(d):
        if d == 1:   return '1 month'
        if d <= 3:   return '2-3 months'
        return '4+ months'

    campaigns_df['duration_bucket'] = campaigns_df['duration_months'].apply(bucket_label)
    bucket_order = ['1 month', '2-3 months', '4+ months']

    def spend_row(df, label, tot):
        s = df['total_incremental_opex']
        n = len(df)
        return (f"  {label:<14} {n:>4}  ({n/tot*100:4.1f}%)  "
                f"mean ${s.mean():>8,.0f}  "
                f"std ${s.std():>8,.0f}  "
                f"p25 ${s.quantile(0.25):>8,.0f}  "
                f"p50 ${s.median():>8,.0f}  "
                f"p75 ${s.quantile(0.75):>8,.0f}")

    print(f"Raw spike months:            {len(spikes)}")
    print(f"Campaigns detected:          {tot}")
    print(f"Unique sites with campaigns: {campaigns_df['site_key'].nunique()}")
    print()
    hdr = (f"  {'Bucket':<14} {'N':>4}   {'%':>6}   "
           f"{'Mean':>12}  {'Std':>12}  {'p25':>12}  {'Median':>12}  {'p75':>12}")
    print(hdr)
    print("  " + "-" * (len(hdr) - 2))
    for b in bucket_order:
        sub = campaigns_df[campaigns_df['duration_bucket'] == b]
        if len(sub) == 0:
            continue
        s = sub['total_incremental_opex']
        n = len(sub)
        print(f"  {b:<14} {n:>4}  ({n/tot*100:4.1f}%)  "
              f"${s.mean():>10,.0f}  "
              f"${s.std():>10,.0f}  "
              f"${s.quantile(0.25):>10,.0f}  "
              f"${s.median():>10,.0f}  "
              f"${s.quantile(0.75):>10,.0f}")
    # All-campaigns row
    s_all = campaigns_df['total_incremental_opex']
    print(f"  {'All':<14} {tot:>4}  (100.0%)  "
          f"${s_all.mean():>10,.0f}  "
          f"${s_all.std():>10,.0f}  "
          f"${s_all.quantile(0.25):>10,.0f}  "
          f"${s_all.median():>10,.0f}  "
          f"${s_all.quantile(0.75):>10,.0f}")
    return campaigns_df

campaigns_df = cluster_campaigns(spikes)


Raw spike months:            244
Campaigns detected:          165
Unique sites with campaigns: 84

  Bucket            N        %           Mean           Std           p25        Median           p75
  ---------------------------------------------------------------------------------------------------
  1 month         123  (74.5%)  $    53,245  $    71,836  $    14,725  $    21,470  $    38,713
  2-3 months       34  (20.6%)  $    79,912  $    80,409  $    29,178  $    54,432  $    94,078
  4+ months         8  ( 4.8%)  $   196,937  $   148,733  $   127,879  $   167,029  $   211,681
  All             165  (100.0%)  $    65,707  $    84,157  $    15,328  $    26,518  $    79,161


In [ ]:
campaigns_df

,site_key,campaign_start,campaign_end,duration_months,total_incremental_opex,peak_opex_ratio,n_spikes,duration_bucket
0,bluewave_000567__1,2024-05-31,2024-05-31,1,11077.558333,1.219945,1,1 month
1,bluewave_000567__1,2024-10-31,2024-12-31,3,73589.953333,1.841472,3,2-3 months
2,bluewave_000567__10,2024-05-31,2024-05-31,1,11594.266667,1.223267,1,1 month
3,bluewave_000567__10,2024-12-31,2024-12-31,1,9707.003333,1.222761,1,1 month
4,bluewave_000567__10,2025-09-30,2025-09-30,1,9969.136667,1.202900,1,1 month
...,...,...,...,...,...,...,...,...
160,vibecw_000482__2,2026-01-31,2026-02-28,2,72937.223333,1.493073,2,2-3 months
161,vibecw_000482__4,2024-12-31,2024-12-31,1,30582.255000,1.391992,1,1 month
162,vibecw_000482__5,2025-05-31,2025-06-30,2,41835.907333,1.320692,2,2-3 months
163,vibecw_000482__5,2026-01-31,2026-01-31,1,27464.740000,1.307637,1,1 month


In [ ]:
campaigns_df['duration_bucket'] = pd.cut(
    campaigns_df['duration_months'],
    bins=[0, 1, 3, 100],
    labels=['1 month', '2-3 months', '4+ months']
)

color_map = {'1 month': '#1565C0', '2-3 months': '#E65100', '4+ months': '#6A1B9A'}

fig = go.Figure()
for bucket, color in color_map.items():
    sub = campaigns_df[campaigns_df['duration_bucket'] == bucket]
    if len(sub) == 0:
        continue
    fig.add_trace(go.Histogram(
        x=sub['total_incremental_opex'],
        name=bucket,
        marker_color=color,
        opacity=0.75,
        nbinsx=25,
    ))

med_spend = campaigns_df['total_incremental_opex'].median()
fig.add_vline(x=med_spend, line=dict(color='#333', dash='dash', width=1.5))
fig.add_annotation(
    x=med_spend, y=1, yref='paper',
    text=f'Median: ${med_spend:,.0f}',
    showarrow=True, arrowhead=2, ay=-30,
    font=dict(size=11), bgcolor='rgba(255,255,255,0.85)',
)
fig.update_layout(
    barmode='overlay',
    title=dict(
        text='<b>Campaign Incremental OPEX Spend Distribution</b>',
            #  '<br><sub>One entry per campaign · colored by duration</sub>',
        x=0.02, xanchor='left',
    ),
    xaxis_title='Incremental OPEX Spend ($)',
    yaxis_title='Number of Campaigns',
    plot_bgcolor='white',
    legend=dict(title='Campaign duration'),
    margin=dict(l=60, r=40, t=100, b=60),
    height=420,
)
fig.show()


In [ ]:
# ── Multi-spike site distribution ────────────────────────────────────────────
spike_counts = spikes.groupby("site_key").size().rename("n_spikes")

# All sites in data, including those with zero spikes
all_sites    = data["site_key"].nunique()
sites_spiked = spike_counts.index.nunique()
sites_zero   = all_sites - sites_spiked

def bucket(n):
    if n == 1: return "1 spike"
    if n <= 3: return "2–3 spikes"
    if n <= 5: return "4–5 spikes"
    return "6+ spikes"

spike_counts_df = spike_counts.reset_index()
spike_counts_df["bucket"] = spike_counts_df["n_spikes"].apply(bucket)

bucket_order = ["1 spike", "2–3 spikes", "4–5 spikes", "6+ spikes"]
bucket_colors = {
    "1 spike":     "#90A4AE",
    "2–3 spikes":  "#1565C0",
    "4–5 spikes":  "#E65100",
    "6+ spikes":   "#6A1B9A",
}

print(f"Total sites in portfolio : {all_sites}")
print(f"Sites with ≥1 OPEX spike : {sites_spiked}  ({sites_spiked/all_sites*100:.1f}%)")
print(f"Sites with 0 spikes      : {sites_zero}  ({sites_zero/all_sites*100:.1f}%)")
print()
print(f"  {'Bucket':<14} {'Sites':>6}  {'%':>6}  {'Avg spikes':>10}  {'Max spikes':>10}")
print("  " + "-"*50)
for b in bucket_order:
    sub = spike_counts_df[spike_counts_df["bucket"] == b]
    if len(sub) == 0:
        continue
    n = len(sub)
    print(f"  {b:<14} {n:>6}  ({n/all_sites*100:4.1f}%)  "
          f"{sub['n_spikes'].mean():>9.1f}  {sub['n_spikes'].max():>10}")

print()
print(f"  {'Total spiking':<14} {sites_spiked:>6}  (100.0%)")

# ── Bar chart: sites by spike count ──────────────────────────────────────────
count_dist = spike_counts.value_counts().sort_index().reset_index()
count_dist.columns = ["n_spikes", "n_sites"]

bar_colors = [
    "#6A1B9A" if n >= 6 else
    "#E65100" if n >= 4 else
    "#1565C0" if n >= 2 else
    "#90A4AE"
    for n in count_dist["n_spikes"]
]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=count_dist["n_spikes"],
    y=count_dist["n_sites"],
    marker_color=bar_colors,
    hovertemplate="Sites with %{x} spike(s): %{y}<extra></extra>",
    text=count_dist["n_sites"],
    textposition="outside",
))

fig.update_layout(
    title=dict(
        text=(
            "<b>Sites by Number of OPEX Spikes</b>"
            f"<br><sub>{all_sites} total sites · {sites_zero} never spiked (grey = 0) · "
            "blue = 2–3 · orange = 4–5 · purple = 6+</sub>"
        ),
        x=0.02, xanchor="left",
    ),
    xaxis=dict(
        title="Number of OPEX Spikes per Site",
        dtick=1, showgrid=False, showline=True, linecolor="#CCC",
    ),
    yaxis=dict(title="Number of Sites", gridcolor="#EEE"),
    plot_bgcolor="white",
    height=400,
    margin=dict(l=70, r=40, t=110, b=60),
    bargap=0.25,
)

# add a zero-spike bar manually
fig.add_trace(go.Bar(
    x=[0], y=[sites_zero],
    marker_color="#CFD8DC",
    hovertemplate=f"Sites with 0 spikes: {sites_zero}<extra></extra>",
    text=[sites_zero], textposition="outside",
    showlegend=False,
))
fig.show()


Total sites in portfolio : 139
Sites with ≥1 OPEX spike : 84  (60.4%)
Sites with 0 spikes      : 55  (39.6%)

  Bucket          Sites       %  Avg spikes  Max spikes
  --------------------------------------------------
  1 spike            25  (18.0%)        1.0           1
  2–3 spikes         38  (27.3%)        2.5           3
  4–5 spikes         11  ( 7.9%)        4.4           5
  6+ spikes          10  ( 7.2%)        7.6           9

  Total spiking      84  (100.0%)


## Section 2 — Campaign Revenue ROI
For each campaign, compute monthly incremental revenue vs. the pre-campaign baseline, then track cumulative revenue recovery vs. the campaign spend. Answers: when does the median campaign break even?

In [ ]:
def compute_campaign_roi(data, campaigns_df, pre_months=6, post_months=12):
    """
    For each campaign (anchored at campaign_start), compute incremental revenue
    vs. pre-campaign baseline and cumulative ROI = cumulative_incr / campaign_spend.
    Returns roi_df with one row per (campaign, month_offset).
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    base_records = []
    for _, campaign in campaigns_df.iterrows():
        site_key       = campaign['site_key']
        anchor_date    = campaign['campaign_start']
        campaign_spend = campaign['total_incremental_opex']
        if campaign_spend <= 0 or site_key not in rd_by_site:
            continue

        ts = rd_by_site[site_key].copy()
        ts['mfc'] = (
            (ts['report_date'].dt.year  - anchor_date.year)  * 12 +
            (ts['report_date'].dt.month - anchor_date.month)
        )
        window = ts[(ts['mfc'] >= -pre_months) & (ts['mfc'] <= post_months)]
        pre    = window[(window['mfc'] >= -pre_months) & (window['mfc'] < 0)]
        if len(pre) < 3:
            continue
        baseline_rev = pre['total_income'].mean()
        if pd.isna(baseline_rev) or baseline_rev <= 0:
            continue

        for _, row in window.iterrows():
            base_records.append({
                'site_key':             site_key,
                'campaign_start':       anchor_date,
                'months_from_campaign': int(row['mfc']),
                'incremental_revenue':  row['total_income'] - baseline_rev,
                'baseline_revenue':     baseline_rev,
                'campaign_spend':       campaign_spend,
            })

    if not base_records:
        print("No campaigns with sufficient revenue data.")
        return pd.DataFrame()

    base_df = pd.DataFrame(base_records)

    # Add cumulative ROI per campaign (running sum from M0 onwards)
    cum_records = []
    for (sk, cs), grp in base_df.groupby(['site_key', 'campaign_start']):
        grp   = grp.sort_values('months_from_campaign')
        spend = grp['campaign_spend'].iloc[0]
        cum_rev = 0.0
        for _, row in grp.iterrows():
            mfc = row['months_from_campaign']
            if mfc >= 0:
                cum_rev += row['incremental_revenue']
            cum_records.append({
                'site_key':                      sk,
                'campaign_start':                cs,
                'months_from_campaign':          mfc,
                'incremental_revenue':           row['incremental_revenue'],
                'baseline_revenue':              row['baseline_revenue'],
                'campaign_spend':                spend,
                'cumulative_incremental_revenue': cum_rev if mfc >= 0 else np.nan,
                'cumulative_roi':                (cum_rev / spend) if (mfc >= 0 and spend > 0) else np.nan,
            })

    roi_df = pd.DataFrame(cum_records)
    n_campaigns = roi_df.groupby(['site_key', 'campaign_start']).ngroups
    print(f"Campaigns with sufficient data for ROI: {n_campaigns}")
    return roi_df

roi_df = compute_campaign_roi(data, campaigns_df)


Campaigns with sufficient data for ROI: 156


In [ ]:
if len(roi_df) == 0:
    print("No ROI data to plot.")
else:
    agg_incr = (
        roi_df.groupby('months_from_campaign')['incremental_revenue']
        .median().reset_index()
    )
    agg_roi = (
        roi_df[roi_df['months_from_campaign'] >= 0]
        .groupby('months_from_campaign')['cumulative_roi']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75))
        .reset_index()
    )

    breakeven     = agg_roi[agg_roi['median'] >= 1.0]
    breakeven_month = int(breakeven['months_from_campaign'].iloc[0]) if len(breakeven) > 0 else None

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Median Incremental Revenue per Month ($) vs. pre-campaign baseline',
            'Median Cumulative ROI (x campaign spend) — how long to break even?',
        ],
        vertical_spacing=0.14,
    )

    # Panel 1: incremental revenue bar chart
    bar_colors = ['#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#9E9E9E')
                  for x in agg_incr['months_from_campaign']]
    fig.add_trace(go.Bar(
        x=agg_incr['months_from_campaign'],
        y=agg_incr['incremental_revenue'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median incr. revenue: $%{y:,.0f}<extra></extra>',
    ), row=1, col=1)
    fig.add_hline(y=0, line=dict(color='#333', dash='dash', width=1), row=1, col=1)
    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5), row=1, col=1)

    # Panel 2: cumulative ROI with IQR band
    x_roi = agg_roi['months_from_campaign']
    fig.add_trace(go.Scatter(
        x=list(x_roi) + list(x_roi[::-1]),
        y=list(agg_roi['q75']) + list(agg_roi['q25'][::-1]),
        fill='toself', fillcolor='#6A1B9A', opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=2, col=1)
    fig.add_trace(go.Scatter(
        x=x_roi, y=agg_roi['median'],
        mode='lines+markers',
        line=dict(color='#6A1B9A', width=2.5), marker=dict(size=7),
        showlegend=False,
        hovertemplate='M+%{x}<br>Cumulative ROI: %{y:.2f}x<extra></extra>',
    ), row=2, col=1)
    fig.add_hline(y=1.0, line=dict(color='#2E7D32', dash='dash', width=1.5), row=2, col=1)
    fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot',  width=1.5), row=2, col=1)
    if breakeven_month is not None:
        fig.add_vline(x=breakeven_month,
                      line=dict(color='#2E7D32', dash='dot', width=1.5), row=2, col=1)
        fig.add_annotation(
            x=breakeven_month + 0.3, y=agg_roi[agg_roi['months_from_campaign'] == breakeven_month]['median'].iloc[0],
            text=f'Break-even: M+{breakeven_month}',
            showarrow=False,
            font=dict(color='#2E7D32', size=11),
            bgcolor='rgba(255,255,255,0.85)',
            row=2, col=1,
        )

    fig.update_yaxes(title_text='Incremental Revenue ($)', tickprefix='$', gridcolor='#EEE', row=1, col=1)
    fig.update_yaxes(title_text='Cumulative ROI (x)', tickformat='.2f', gridcolor='#EEE', row=2, col=1)
    for r in [1, 2]:
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=r, col=1)
    fig.update_xaxes(title_text='Months from Campaign Start', row=2, col=1)

    be_txt = (f'Median break-even: M+{breakeven_month}'
              if breakeven_month is not None else 'Break-even not reached within window')
    fig.update_layout(
        title=dict(
            text=f'<b>Campaign Revenue ROI</b><br><sub>{be_txt} · shaded band = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        showlegend=False,
        plot_bgcolor='white',
        height=640,
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.show()


In [ ]:
if len(roi_df) == 0:
    print("No ROI data to plot.")
else:
    # total_revenue_norm = actual / baseline  (1.0 = flat at baseline)
    roi_df['total_revenue_norm'] = (
        (roi_df['incremental_revenue'] + roi_df['baseline_revenue']) / roi_df['baseline_revenue']
    )

    agg_norm = (
        roi_df.groupby('months_from_campaign')['total_revenue_norm']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )

    bar_colors = [
        '#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#90A4AE')
        for x in agg_norm['months_from_campaign']
    ]

    fig = go.Figure()

    # IQR band
    x = agg_norm['months_from_campaign']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg_norm['q75']) + list(agg_norm['q25'][::-1]),
        fill='toself', fillcolor='#1565C0', opacity=0.10,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))

    # Median bars
    fig.add_trace(go.Bar(
        x=x,
        y=agg_norm['median'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median total revenue: %{y:.2f}× baseline<extra></extra>',
        name='Median total revenue (normalized)',
    ))

    fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1.5))
    fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5))

    n_campaigns = roi_df.groupby(['site_key', 'campaign_start']).ngroups
    fig.update_layout(
        title=dict(
            text=('<b>Total Revenue MoM Around Promotional Spike</b>'
                  f'<br><sub>{n_campaigns} campaigns · 1.0 = pre-campaign baseline · ' 
                  'red dotted = campaign start · shaded = 25–75th percentile</sub>'),
            x=0.02, xanchor='left',
        ),
        xaxis_title='Months from Campaign Start',
        yaxis_title='Total Revenue (× baseline)',
        yaxis=dict(tickformat='.2f', gridcolor='#EEE'),
        xaxis=dict(showgrid=False, showline=True, linecolor='#CCC'),
        plot_bgcolor='white',
        showlegend=False,
        height=420,
        margin=dict(l=80, r=40, t=110, b=60),
    )
    fig.show()


In [ ]:
def compute_total_revenue_by_spike(data, spikes, pre_months=6, post_months=12, min_pre_obs=3):
    """
    Anchors on every raw OPEX spike (all 264 spike-months).
    No clustering — each spike month is an independent event anchor.
    Returns a DataFrame with median total_income in $ at each month offset.
    Dropped only when a site has < min_pre_obs months before the spike.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    records = []
    skipped = 0
    for _, spike in spikes.iterrows():
        site_key   = spike['site_key']
        spike_date = pd.to_datetime(spike['report_date'])
        if site_key not in rd_by_site:
            skipped += 1
            continue

        ts = rd_by_site[site_key].copy()
        ts['mfs'] = (
            (ts['report_date'].dt.year  - spike_date.year)  * 12 +
            (ts['report_date'].dt.month - spike_date.month)
        )
        window = ts[(ts['mfs'] >= -pre_months) & (ts['mfs'] <= post_months)]
        pre    = window[window['mfs'] < 0]
        if len(pre) < min_pre_obs:
            skipped += 1
            continue

        for _, row in window.iterrows():
            records.append({
                'site_key':             site_key,
                'spike_date':           spike_date,
                'months_from_spike':    int(row['mfs']),
                'total_income':         row['total_income'],
            })

    rev_spike_df = pd.DataFrame(records)
    n_spikes_used = rev_spike_df.groupby(['site_key', 'spike_date']).ngroups
    print(f"Total raw spikes    : {len(spikes)}")
    print(f"Spikes with data    : {n_spikes_used}  (dropped {skipped} — insufficient pre-spike history)")
    return rev_spike_df

rev_spike_df = compute_total_revenue_by_spike(data, spikes)


Total raw spikes    : 244
Spikes with data    : 233  (dropped 2 — insufficient pre-spike history)


In [ ]:
if len(rev_spike_df) == 0:
    print("No spike revenue data to plot.")
else:
    agg = (
        rev_spike_df.groupby('months_from_spike')['total_income']
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )

    bar_colors = [
        '#C62828' if x == 0 else ('#2E7D32' if x > 0 else '#90A4AE')
        for x in agg['months_from_spike']
    ]

    fig = go.Figure()

    # IQR band
    x = agg['months_from_spike']
    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor='#1565C0', opacity=0.10,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ))

    # Median bars
    fig.add_trace(go.Bar(
        x=x,
        y=agg['median'],
        marker_color=bar_colors,
        hovertemplate='M%{x}<br>Median total revenue: $%{y:,.0f}<extra></extra>',
    ))

    fig.add_vline(x=0, line=dict(color='#C62828', dash='dot', width=1.5))

    n_spikes_used = rev_spike_df.groupby(['site_key', 'spike_date']).ngroups
    fig.update_layout(
        title=dict(
            text=('<b>Total Revenue ($) by Month — All OPEX Spikes</b>'
                  f'<br><sub>{n_spikes_used} spike events · actual median $ · ' 
                  'red dotted = spike month · shaded = 25–75th percentile</sub>'),
            x=0.02, xanchor='left',
        ),
        xaxis_title='Months from OPEX Spike',
        yaxis_title='Median Total Revenue ($)',
        yaxis=dict(tickprefix='$', tickformat=',.0f', gridcolor='#EEE'),
        xaxis=dict(showgrid=False, showline=True, linecolor='#CCC'),
        plot_bgcolor='white',
        showlegend=False,
        height=420,
        margin=dict(l=90, r=40, t=110, b=60),
    )
    fig.show()


In [ ]:
# ── OPEX / Revenue / Profit / Mem Purchases — 3 separate plots ───────────────
_rd = data.copy()
_rd["report_date"] = pd.to_datetime(_rd["report_date"])
_rd["true_opex"]   = _rd["cogs"] + _rd["expenses"]
_rd_by_site = {sk: grp.sort_values("report_date")
               for sk, grp in _rd.groupby("site_key")}

def reclassify(d):
    if d == 1: return "1 month"
    if d == 2: return "2 months"
    return "3+ months"

campaigns_df["dur_bucket2"] = campaigns_df["duration_months"].apply(reclassify)

_records = []
for _, camp in campaigns_df.iterrows():
    sk     = camp["site_key"]
    anchor = pd.to_datetime(camp["campaign_start"])
    bucket = camp["dur_bucket2"]
    dur    = int(camp["duration_months"])
    if sk not in _rd_by_site:
        continue
    ts = _rd_by_site[sk].copy()
    ts["mfs"] = (
        (ts["report_date"].dt.year  - anchor.year)  * 12 +
        (ts["report_date"].dt.month - anchor.month)
    )
    for _, row in ts[(ts["mfs"] >= -3) & (ts["mfs"] <= 6)].iterrows():
        _records.append({
            "bucket":           bucket,
            "duration":         dur,
            "mfs":              int(row["mfs"]),
            "opex":             row["true_opex"],
            "revenue":          row["total_income"],
            "profit":           row["total_income"] - row["true_opex"],
            "mem_purchases":    row["mem_purchase_count"],
        })

snap_df2 = pd.DataFrame(_records)

OPEX_COLOR    = "#4FC3F7"
REV_COLOR     = "#FFA726"
PROFIT_COLOR  = "#66BB6A"
MEM_COLOR     = "#CE93D8"   # purple — membership purchases
CAMP_SHADE    = "#FFF9C4"

BUCKET_CONFIG = {
    "1 month":   {"camp_months": [0],       "title": "1-Month Campaigns"},
    "2 months":  {"camp_months": [0, 1],    "title": "2-Month Campaigns"},
    "3+ months": {"camp_months": [0, 1, 2], "title": "3+ Month Campaigns"},
}

for bucket, cfg in BUCKET_CONFIG.items():
    sub = snap_df2[snap_df2["bucket"] == bucket]
    if len(sub) == 0:
        continue

    agg = (
        sub.groupby("mfs")[["opex", "revenue", "profit", "mem_purchases"]]
        .median()
        .reset_index()
    )
    n_camps     = campaigns_df[campaigns_df["dur_bucket2"] == bucket].shape[0]
    camp_months = cfg["camp_months"]
    month_list  = list(agg["mfs"])

    x_labels = []
    for m in month_list:
        if m in camp_months:
            x_labels.append(f"T={m}<br><b>📍 Campaign</b>")
        else:
            x_labels.append(f"T={m}")

    fig = go.Figure()

    # Shade campaign months
    for cm in camp_months:
        if cm in month_list:
            pos = month_list.index(cm)
            fig.add_vrect(
                x0=pos - 0.5, x1=pos + 0.5,
                fillcolor=CAMP_SHADE, opacity=0.55,
                layer="below", line_width=0,
            )

    fig.add_trace(go.Bar(
        name="OPEX", x=x_labels, y=agg["opex"],
        marker_color=OPEX_COLOR,
        hovertemplate="T=%{x}<br>OPEX: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Revenue", x=x_labels, y=agg["revenue"],
        marker_color=REV_COLOR,
        hovertemplate="T=%{x}<br>Revenue: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Profit (Rev − OPEX)", x=x_labels, y=agg["profit"],
        marker_color=PROFIT_COLOR,
        hovertemplate="T=%{x}<br>Profit: $%{y:,.0f}<extra></extra>",
    ))
    fig.add_trace(go.Bar(
        name="Membership Purchases", x=x_labels, y=agg["mem_purchases"],
        marker_color=MEM_COLOR,
        hovertemplate="T=%{x}<br>Mem Purchases: %{y:,.0f}<extra></extra>",
    ))

    camp_label = (
        "T=0 = campaign month"
        if len(camp_months) == 1
        else f"T={camp_months[0]}–T={camp_months[-1]} = campaign months (yellow)"
    )

    fig.update_layout(
        barmode="group",
        title=dict(
            text=(
                f"<b>OPEX · Revenue · Profit · Membership Purchases — {cfg['title']}</b>"
                f"<br><sub>n={n_camps} campaigns · {camp_label} · "
                "median values · T=-3 to T=-1 = pre-campaign baseline</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        xaxis=dict(
            title="Month Offset from Campaign Start",
            showgrid=False, showline=True, linecolor="#CCC",
        ),
        yaxis=dict(
            title="Median Value",
            tickformat=",.0f",
            gridcolor="#EEE",
        ),
        legend=dict(orientation="h", x=0.02, y=1.01, xanchor="left"),
        plot_bgcolor="white",
        height=470,
        margin=dict(l=90, r=40, t=120, b=60),
        bargap=0.18,
        bargroupgap=0.04,
    )
    fig.show()


In [ ]:
# ── Statistical test: hypothesis — post-campaign OPEX drop → Profit & Membership rise ──
# H0: no difference between campaign month and first post-campaign month
# H1 (one-sided): Profit rises AND Membership Purchases rise when OPEX drops post-campaign
# Test: Wilcoxon signed-rank (non-parametric, robust to skewed financial data)
# Paired at the campaign level: (last campaign month) vs (first post-campaign month)

from scipy import stats as _stats

_rd = data.copy()
_rd["report_date"] = pd.to_datetime(_rd["report_date"])
_rd["true_opex"]   = _rd["cogs"] + _rd["expenses"]
_rd["profit"]      = _rd["total_income"] - _rd["true_opex"]
_rd_by_site = {sk: grp.sort_values("report_date")
               for sk, grp in _rd.groupby("site_key")}

def reclassify(d):
    if d == 1: return "1 month"
    if d == 2: return "2 months"
    return "3+ months"

campaigns_df["dur_bucket2"] = campaigns_df["duration_months"].apply(reclassify)

# Build paired observations per campaign
# "campaign month" = last spike month (T = duration-1 from anchor)
# "post-campaign month" = T = duration (first month after campaign ends)
paired_records = []
for _, camp in campaigns_df.iterrows():
    sk      = camp["site_key"]
    anchor  = pd.to_datetime(camp["campaign_start"])
    dur     = int(camp["duration_months"])
    bucket  = camp["dur_bucket2"]
    if sk not in _rd_by_site:
        continue
    ts = _rd_by_site[sk].copy()
    ts["mfs"] = (
        (ts["report_date"].dt.year  - anchor.year)  * 12 +
        (ts["report_date"].dt.month - anchor.month)
    )
    # campaign month = last month of campaign (mfs = dur - 1)
    # post-campaign  = first month after (mfs = dur)
    t_camp = dur - 1
    t_post = dur
    camp_row = ts[ts["mfs"] == t_camp]
    post_row = ts[ts["mfs"] == t_post]
    if len(camp_row) == 0 or len(post_row) == 0:
        continue
    paired_records.append({
        "bucket":           bucket,
        "duration":         dur,
        # campaign month values
        "opex_camp":        camp_row["true_opex"].iloc[0],
        "profit_camp":      camp_row["profit"].iloc[0],
        "mem_camp":         camp_row["mem_purchase_count"].iloc[0],
        "revenue_camp":     camp_row["total_income"].iloc[0],
        # post-campaign month values
        "opex_post":        post_row["true_opex"].iloc[0],
        "profit_post":      post_row["profit"].iloc[0],
        "mem_post":         post_row["mem_purchase_count"].iloc[0],
        "revenue_post":     post_row["total_income"].iloc[0],
    })

pairs_df = pd.DataFrame(paired_records)

BUCKET_ORDER = ["1 month", "2 months", "3+ months"]

def run_wilcoxon(before, after, direction="greater"):
    """Wilcoxon signed-rank: tests if after > before (one-sided)."""
    diff = after - before
    diff = diff.dropna()
    if len(diff) < 10:
        return np.nan, np.nan, len(diff)
    stat, p = _stats.wilcoxon(diff, alternative=direction)
    return stat, p, len(diff)

def pct_improved(before, after):
    diff = after - before
    return (diff > 0).sum() / len(diff.dropna()) * 100

print("=" * 80)
print("  HYPOTHESIS TEST")
print("  'When OPEX drops after campaign month, Profit and Membership Purchases rise'")
print("  Method: Wilcoxon signed-rank test (one-sided) | α = 0.05")
print("=" * 80)

for bucket in BUCKET_ORDER:
    sub = pairs_df[pairs_df["bucket"] == bucket]
    if len(sub) < 5:
        continue

    print(f"\n{'─'*80}")
    print(f"  Bucket: {bucket}  (n={len(sub)} campaigns)")
    print(f"{'─'*80}")

    print(f"  {'Metric':<28} {'Camp median':>12} {'Post median':>12} "
          f"{'% improved':>11} {'p-value':>9} {'Sig':>4}")
    print(f"  {'-'*74}")

    tests = [
        ("OPEX ($)",            "opex_camp",    "opex_post",    "less"),
        ("Revenue ($)",         "revenue_camp", "revenue_post", "greater"),
        ("Profit ($)",          "profit_camp",  "profit_post",  "greater"),
        ("Mem Purchases (ct)",  "mem_camp",     "mem_post",     "greater"),
    ]

    for label, col_b, col_a, direction in tests:
        before = sub[col_b]
        after  = sub[col_a]
        stat, p, n = run_wilcoxon(before, after, direction)
        pct = pct_improved(before, after)
        m_before = before.median()
        m_after  = after.median()
        if pd.isna(p):
            sig = "n/a"
            p_str = "n/a"
        else:
            sig   = "✓**" if p < 0.01 else ("✓*" if p < 0.05 else "✗")
            p_str = f"{p:.4f}"

        # For OPEX: "improved" means it went DOWN
        pct_display = pct if direction == "greater" else (100 - pct)
        prefix = "$" if "($)" in label else ""
        print(f"  {label:<28} {prefix}{m_before:>10,.0f}   {prefix}{m_after:>10,.0f}  "
              f"{pct_display:>9.1f}%  {p_str:>9}  {sig:>4}")

print(f"\n{'─'*80}")
print("  Significance: ✓** p<0.01  ✓* p<0.05  ✗ not significant")
print("  OPEX '% improved' = % of campaigns where OPEX fell post-campaign (expected)")
print("  Profit/Mem/Revenue '% improved' = % where the metric rose post-campaign")
print(f"{'─'*80}")


  HYPOTHESIS TEST
  'When OPEX drops after campaign month, Profit and Membership Purchases rise'
  Method: Wilcoxon signed-rank test (one-sided) | α = 0.05

────────────────────────────────────────────────────────────────────────────────
  Bucket: 1 month  (n=96 campaigns)
────────────────────────────────────────────────────────────────────────────────
  Metric                        Camp median  Post median  % improved   p-value  Sig
  --------------------------------------------------------------------------
  OPEX ($)                     $    92,328   $    74,558       97.9%     0.0000   ✓**
  Revenue ($)                  $   118,174   $   110,145       37.5%     0.9985     ✗
  Profit ($)                   $    31,320   $    45,497       64.6%     0.0003   ✓**
  Mem Purchases (ct)                2,546        2,635       57.3%     0.0251    ✓*

────────────────────────────────────────────────────────────────────────────────
  Bucket: 2 months  (n=22 campaigns)
───────────────────────

## Section 3 — Site Ramp Trajectory
Track OPEX spend, membership growth, and market share from a site's first month in the data. Aggregated across all 162 sites — shows the typical trajectory over the first 18 months.

In [ ]:
def analyze_new_site_ramp(data, dist_df, max_age_months=18, min_months_data=6):
    """
    For each site, track OPEX, membership washes, and market share as a function
    of months since first appearance in data. Aggregate across all sites.
    """
    rd = data.copy()
    rd['report_date']  = pd.to_datetime(rd['report_date'])
    rd['true_opex']    = rd['cogs'] + rd['expenses']
    rd['total_washes'] = rd['mem_wash_count'].fillna(0) + rd['ret_wash_count'].fillna(0)

    # Fast lookup: date x site_key → total_washes
    washes_pivot = rd.pivot_table(
        index='report_date', columns='site_key', values='total_washes', aggfunc='sum'
    )

    first_reports = rd.groupby('site_key')['report_date'].min().to_dict()

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(grp['neighbor_key'])

    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    records = []
    for site_key, ts in rd_by_site.items():
        first_date = first_reports[site_key]
        ts = ts.copy()
        ts['age_months'] = (
            (ts['report_date'].dt.year  - first_date.year)  * 12 +
            (ts['report_date'].dt.month - first_date.month) + 1
        )
        ts = ts[(ts['age_months'] >= 1) & (ts['age_months'] <= max_age_months)]
        if len(ts) < min_months_data:
            continue

        nbrs = nbr_lookup.get(site_key, [])

        for _, row in ts.iterrows():
            date     = row['report_date']
            f_washes = row['total_washes']

            mkt_share = np.nan
            if nbrs and date in washes_pivot.index:
                valid_nbrs = [n for n in nbrs if n in washes_pivot.columns]
                if valid_nbrs:
                    nbr_washes = washes_pivot.loc[date, valid_nbrs].fillna(0).sum()
                    mkt        = f_washes + nbr_washes
                    mkt_share  = f_washes / mkt if mkt > 0 else np.nan

            records.append({
                'site_key':       site_key,
                'age_months':     int(row['age_months']),
                'opex':           row['true_opex'],
                'mem_wash_count': row['mem_wash_count'],
                'total_washes':   f_washes,
                'market_share':   mkt_share,
                'has_neighbors':  len(nbrs) > 0,
            })

    ramp_df = pd.DataFrame(records)
    n_sites = ramp_df['site_key'].nunique()
    print(f"Sites in ramp analysis: {n_sites}")
    for m in [6, 12, 18]:
        ms = ramp_df[ramp_df['age_months'] == m]['market_share'].dropna()
        if len(ms) >= 5:
            print(f"  Median market share at month {m:2d}: {ms.median()*100:.1f}%  (n={len(ms)} sites with neighbors)")
    return ramp_df

ramp_df = analyze_new_site_ramp(data, dist_df)


Sites in ramp analysis: 110
  Median market share at month  6: 28.6%  (n=85 sites with neighbors)
  Median market share at month 12: 28.7%  (n=67 sites with neighbors)
  Median market share at month 18: 29.0%  (n=50 sites with neighbors)


In [ ]:
RAMP_COLOR = '#1565C0'

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[
        'OPEX ($) — median across all sites',
        'Membership Wash Count — median across all sites',
        'Market Share within 20 km (%) — sites with neighbors only',
    ],
    vertical_spacing=0.08,
)

specs = [
    ('opex',           'OPEX ($)',          False, ramp_df),
    ('mem_wash_count', 'Membership washes', False, ramp_df),
    ('market_share',   'Market share (%)',  True,  ramp_df[ramp_df['has_neighbors']]),
]

for i, (col, ylabel, is_pct, subset) in enumerate(specs, start=1):
    sub = subset.copy()
    if is_pct:
        sub[col] = sub[col] * 100

    agg = (
        sub.groupby('age_months')[col]
        .agg(median='median',
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n='count')
        .reset_index()
    )
    agg = agg[agg['n'] >= 5]
    x   = agg['age_months']

    fig.add_trace(go.Scatter(
        x=list(x) + list(x[::-1]),
        y=list(agg['q75']) + list(agg['q25'][::-1]),
        fill='toself', fillcolor=RAMP_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo='skip',
    ), row=i, col=1)
    fig.add_trace(go.Scatter(
        x=x, y=agg['median'],
        mode='lines+markers',
        line=dict(color=RAMP_COLOR, width=2.5), marker=dict(size=6),
        showlegend=False,
        hovertemplate=f'Month %{{x}}<br>Median: %{{y:,.1f}}<extra></extra>',
    ), row=i, col=1)

    suffix = '%' if is_pct else ''
    fig.update_yaxes(title_text=ylabel, gridcolor='#EEE', ticksuffix=suffix, row=i, col=1)
    fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

n_sites = ramp_df['site_key'].nunique()
fig.update_xaxes(title_text='Months since first report', row=3, col=1)
fig.update_layout(
    title=dict(
        text=f'<b>Site Ramp Trajectory — First 18 Months in Data</b>'
             f'<br><sub>{n_sites} sites · shaded = 25–75th percentile</sub>',
        x=0.02, xanchor='left',
    ),
    plot_bgcolor='white',
    height=760,
    margin=dict(l=80, r=40, t=110, b=60),
)
fig.show()


## Section 4 — Incumbent Response to New Entrants
When a new site appears in the data, do established incumbents within 20 km react by changing their OPEX spend or losing retail volume? Incumbents are sites with ≥ 6 months of data before the new site's first report.

In [ ]:
def analyze_incumbent_response(data, dist_df, pre_months=3, post_months=12,
                                min_pre_obs=3, incumbent_min_age=6):
    """
    For each site's first report month (treated as 'opening'), measure how
    established incumbents within 20 km respond in OPEX and retail washes.
    Incumbents must have >= incumbent_min_age months of data before the opening.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd['true_opex']   = rd['cogs'] + rd['expenses']

    first_reports = rd.groupby('site_key')['report_date'].min().to_dict()
    rd_by_site    = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(zip(grp['neighbor_key'], grp['distance_km']))

    records = []
    for new_site, opening_date in first_reports.items():
        neighbors = nbr_lookup.get(new_site, [])
        if not neighbors:
            continue

        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_first = first_reports[nbr_key]
            months_established = (
                (opening_date.year  - nbr_first.year)  * 12 +
                (opening_date.month - nbr_first.month)
            )
            if months_established < incumbent_min_age:
                continue

            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['mfs'] = (
                (nbr_ts['report_date'].dt.year  - opening_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - opening_date.month)
            )
            window = nbr_ts[(nbr_ts['mfs'] >= -pre_months) & (nbr_ts['mfs'] <= post_months)]
            pre    = window[window['mfs'] < 0]
            if len(pre) < min_pre_obs:
                continue

            baseline_opex = pre['true_opex'].mean()
            baseline_ret  = pre['ret_wash_count'].mean()
            if pd.isna(baseline_opex) or baseline_opex <= 0:
                continue

            for _, row in window.iterrows():
                records.append({
                    'new_site':            new_site,
                    'incumbent_key':       nbr_key,
                    'distance_km':         dist_km,
                    'months_from_opening': int(row['mfs']),
                    'opex_norm':           row['true_opex'] / baseline_opex,
                    'ret_wash_norm':       (row['ret_wash_count'] / max(baseline_ret, 1)
                                           if pd.notna(row['ret_wash_count']) else np.nan),
                })

    resp_df = pd.DataFrame(records)
    if len(resp_df) > 0:
        n_pairs = resp_df.groupby(['new_site', 'incumbent_key']).ngroups
        print(f"Incumbent-opening pairs: {n_pairs}")
    else:
        print("No incumbent-opening pairs found.")
    return resp_df

resp_df = analyze_incumbent_response(data, dist_df)


Incumbent-opening pairs: 108


In [ ]:
if len(resp_df) == 0:
    print("No incumbent-opening pairs — cannot plot.")
else:
    INC_COLOR = '#C62828'

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Incumbent OPEX (normalized)',
            'Incumbent Retail Wash Count (normalized)',
        ],
        vertical_spacing=0.10,
    )

    for i, col in enumerate(['opex_norm', 'ret_wash_norm'], start=1):
        agg = (
            resp_df.groupby('months_from_opening')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        agg = agg[agg['n'] >= 5]
        x   = agg['months_from_opening']

        fig.add_trace(go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(agg['q75']) + list(agg['q25'][::-1]),
            fill='toself', fillcolor=INC_COLOR, opacity=0.12,
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=i, col=1)
        fig.add_trace(go.Scatter(
            x=x, y=agg['median'],
            mode='lines+markers',
            line=dict(color=INC_COLOR, width=2.5), marker=dict(size=6),
            showlegend=False,
            hovertemplate='M%{x}<br>Median: %{y:.2f}× baseline<extra></extra>',
        ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#1565C0', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f', gridcolor='#EEE', row=i, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

    n_pairs = resp_df.groupby(['new_site', 'incumbent_key']).ngroups
    fig.update_xaxes(title_text='Months from new site opening', row=2, col=1)
    fig.update_layout(
        title=dict(
            text='<b>Incumbent Response to New Entrant Opening</b>'
                 f'<br><sub>{n_pairs} incumbent-opening pairs · '
                 'blue dotted = new site opens · dashed = pre-opening baseline · '
                 'shaded = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        plot_bgcolor='white',
        height=620,
        margin=dict(l=70, r=40, t=120, b=60),
    )
    fig.show()


In [ ]:
def build_entrant_vs_incumbent_panel(data, dist_df,
                                      pre_months=6, post_months=12,
                                      entrant_norm_window=(1, 6),
                                      min_pre_obs=3, incumbent_min_age=6):
    rd = data.copy()
    rd["report_date"]  = pd.to_datetime(rd["report_date"])
    rd["true_opex"]    = rd["cogs"] + rd["expenses"]
    rd["total_washes"] = rd["mem_wash_count"].fillna(0) + rd["ret_wash_count"].fillna(0)

    METRICS = ["true_opex", "ASP_mem", "ASP_ret",
               "total_income", "mem_wash_count", "ret_wash_count", "total_washes"]

    # O(1) wash lookup: report_date × site_key
    washes_pivot = rd.pivot_table(
        index="report_date", columns="site_key", values="total_washes", aggfunc="sum"
    )

    first_reports = rd.groupby("site_key")["report_date"].min().to_dict()
    rd_by_site    = {sk: grp.sort_values("report_date").reset_index(drop=True)
                     for sk, grp in rd.groupby("site_key")}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby("focal_key"):
        nbr_lookup[focal] = list(zip(grp["neighbor_key"], grp["distance_km"]))

    # All sites within 20km of each site (neighbors + self)
    market_sites = {}
    for focal, nbrs in nbr_lookup.items():
        market_sites[focal] = [focal] + [n for n, _ in nbrs]

    entrant_records = []
    incumb_records  = []

    for new_site, opening_date in first_reports.items():
        neighbors = nbr_lookup.get(new_site, [])
        if not neighbors or new_site not in rd_by_site:
            continue

        # ── New entrant ───────────────────────────────────────────────────
        ne_ts = rd_by_site[new_site].copy()
        ne_ts["mfo"] = (
            (ne_ts["report_date"].dt.year  - opening_date.year)  * 12 +
            (ne_ts["report_date"].dt.month - opening_date.month)
        )
        ne_window = ne_ts[(ne_ts["mfo"] >= 0) & (ne_ts["mfo"] <= post_months)]
        lo, hi = entrant_norm_window
        ne_base_rows = ne_ts[(ne_ts["mfo"] >= lo) & (ne_ts["mfo"] <= hi)]
        if len(ne_base_rows) < 3:
            continue

        all_market_sites = market_sites.get(new_site, [new_site])

        for _, row in ne_window.iterrows():
            rec = {"new_site": new_site, "months_from_opening": int(row["mfo"])}
            for m in METRICS:
                rec[m] = row[m]
            # market share: new site washes / total washes of all sites within 20km
            dt = row["report_date"]
            if dt in washes_pivot.index:
                row_washes = washes_pivot.loc[dt]
                mkt_total  = row_washes.reindex(all_market_sites).sum(min_count=1)
                ne_washes  = row_washes.get(new_site, np.nan)
                rec["market_share"] = (ne_washes / mkt_total * 100
                                       if pd.notna(mkt_total) and mkt_total > 0
                                       else np.nan)
            else:
                rec["market_share"] = np.nan
            entrant_records.append(rec)

        # ── Incumbents ────────────────────────────────────────────────────
        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_first = first_reports.get(nbr_key)
            if nbr_first is None:
                continue
            months_estab = (
                (opening_date.year  - nbr_first.year)  * 12 +
                (opening_date.month - nbr_first.month)
            )
            if months_estab < incumbent_min_age:
                continue

            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts["mfo"] = (
                (nbr_ts["report_date"].dt.year  - opening_date.year)  * 12 +
                (nbr_ts["report_date"].dt.month - opening_date.month)
            )
            window = nbr_ts[
                (nbr_ts["mfo"] >= -pre_months) & (nbr_ts["mfo"] <= post_months)
            ]
            pre = window[window["mfo"] < 0]
            if len(pre) < min_pre_obs:
                continue
            if pd.isna(pre["total_income"].mean()) or pre["total_income"].mean() <= 0:
                continue

            for _, row in window.iterrows():
                rec = {
                    "new_site":            new_site,
                    "incumbent_key":       nbr_key,
                    "distance_km":         dist_km,
                    "months_from_opening": int(row["mfo"]),
                }
                for m in METRICS:
                    rec[m] = row[m]
                incumb_records.append(rec)

    entrant_df = pd.DataFrame(entrant_records)
    incumb_df  = pd.DataFrame(incumb_records)
    n_pairs    = incumb_df.groupby(["new_site", "incumbent_key"]).ngroups if len(incumb_df) else 0
    print(f"New sites tracked      : {entrant_df['new_site'].nunique() if len(entrant_df) else 0}")
    print(f"Incumbent-site pairs   : {n_pairs}")
    return entrant_df, incumb_df

entrant_df, incumb_df = build_entrant_vs_incumbent_panel(data, dist_df)


New sites tracked      : 81
Incumbent-site pairs   : 55


In [ ]:
if len(entrant_df) == 0 or len(incumb_df) == 0:
    print("Insufficient data for entrant vs incumbent panel.")
else:
    METRICS_META = [
        ("true_opex",      "OPEX ($)",               "$"),
        ("ASP_mem",        "ASP — Membership ($)",   "$"),
        ("ASP_ret",        "ASP — Retail ($)",       "$"),
        ("total_income",   "Total Revenue ($)",      "$"),
        ("mem_wash_count", "Membership Wash Count",  ""),
        ("ret_wash_count", "Retail Wash Count",      ""),
        ("total_washes",   "Total Wash Count",       ""),
    ]

    NE_COLOR  = "#1565C0"
    INC_COLOR = "#E65100"

    fig = make_subplots(
        rows=8, cols=1,
        shared_xaxes=True,
        subplot_titles=[m[1] for m in METRICS_META] + ["New Entrant Market Share (%)"],
        vertical_spacing=0.04,
    )

    # Panels 1-7: entrant vs incumbent raw metrics
    for row_idx, (col, label, prefix) in enumerate(METRICS_META, start=1):

        for df, color, series_name in [
            (entrant_df, NE_COLOR,  "New Entrant"),
            (incumb_df,  INC_COLOR, "Incumbent"),
        ]:
            grp_col = "months_from_opening"
            agg = (
                df.groupby(grp_col)[col]
                .agg(median="median",
                     q25=lambda s: s.quantile(0.25),
                     q75=lambda s: s.quantile(0.75),
                     n="count")
                .reset_index()
            )
            agg = agg[agg["n"] >= 5]
            x = agg[grp_col]

            fig.add_trace(go.Scatter(
                x=list(x) + list(x[::-1]),
                y=list(agg["q75"]) + list(agg["q25"][::-1]),
                fill="toself", fillcolor=color, opacity=0.10,
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=row_idx, col=1)
            fig.add_trace(go.Scatter(
                x=x, y=agg["median"],
                mode="lines+markers",
                name=series_name if row_idx == 1 else None,
                legendgroup=series_name, showlegend=(row_idx == 1),
                line=dict(color=color, width=2.5), marker=dict(size=5),
                hovertemplate=(
                    f"M%{{x}}<br>{label}: {prefix}%{{y:,.0f}}<extra>{series_name}</extra>"
                    if prefix else
                    f"M%{{x}}<br>{label}: %{{y:,.0f}}<extra>{series_name}</extra>"
                ),
            ), row=row_idx, col=1)

        fig.add_vline(x=0, line=dict(color="#1B5E20", dash="dot", width=1.5),
                      row=row_idx, col=1)
        y_fmt = "$.3s" if prefix == "$" else ","
        fig.update_yaxes(title_text=label, tickformat=y_fmt,
                         gridcolor="#EEE", row=row_idx, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor="#CCC",
                         row=row_idx, col=1)

    # Panel 8: market share (new entrant only — incumbents have no single share)
    agg_ms = (
        entrant_df.groupby("months_from_opening")["market_share"]
        .agg(median="median",
             q25=lambda s: s.quantile(0.25),
             q75=lambda s: s.quantile(0.75),
             n="count")
        .reset_index()
    )
    agg_ms = agg_ms[agg_ms["n"] >= 5]
    x_ms = agg_ms["months_from_opening"]

    fig.add_trace(go.Scatter(
        x=list(x_ms) + list(x_ms[::-1]),
        y=list(agg_ms["q75"]) + list(agg_ms["q25"][::-1]),
        fill="toself", fillcolor=NE_COLOR, opacity=0.12,
        line=dict(width=0), showlegend=False, hoverinfo="skip",
    ), row=8, col=1)
    fig.add_trace(go.Scatter(
        x=x_ms, y=agg_ms["median"],
        mode="lines+markers",
        name="Market Share" if True else None,
        legendgroup="ms", showlegend=True,
        line=dict(color=NE_COLOR, width=2.5), marker=dict(size=5),
        hovertemplate="M%{x}<br>Market share: %{y:.1f}%<extra>New Entrant</extra>",
    ), row=8, col=1)

    fig.add_vline(x=0, line=dict(color="#1B5E20", dash="dot", width=1.5),
                  row=8, col=1)
    fig.update_yaxes(title_text="Market Share (%)", ticksuffix="%",
                     gridcolor="#EEE", row=8, col=1)
    fig.update_xaxes(showgrid=False, showline=True, linecolor="#CCC",
                     title_text="Months from New Entrant Opening", row=8, col=1)

    n_sites = entrant_df["new_site"].nunique()
    n_pairs = incumb_df.groupby(["new_site", "incumbent_key"]).ngroups
    fig.update_layout(
        title=dict(
            text=(
                "<b>New Entrant vs Incumbent — 8 Metrics Around Opening</b>"
                f"<br><sub>{n_sites} new sites · {n_pairs} incumbent pairs · "
                "blue = new entrant · orange = incumbent · "
                "shaded = 25–75th percentile · green dotted = opening month</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        legend=dict(orientation="h", x=0.02, y=1.012, xanchor="left"),
        plot_bgcolor="white",
        height=1900,
        margin=dict(l=110, r=40, t=130, b=60),
    )
    fig.show()


## Section 5 — Neighbor Response to Promotional Spikes
Same structure as Section 4 but the event is a **focal site's OPEX spike** (not a new site opening). For every neighbor within 20km, how does their OPEX and retail wash count change around the spike? No age filter — all neighbors are already established sites.

In [ ]:
def analyze_incumbent_response_to_spikes(data, spikes, dist_df,
                                          pre_months=6, post_months=12, min_pre_obs=3):
    """
    For each OPEX spike at a focal site, measure how neighbors within 20km
    respond in OPEX and retail washes. Normalized by each neighbor's own
    pre-spike baseline. All neighbors qualify — no age filter needed.
    """
    rd = data.copy()
    rd['report_date'] = pd.to_datetime(rd['report_date'])
    rd['true_opex']   = rd['cogs'] + rd['expenses']

    rd_by_site = {sk: grp.sort_values('report_date') for sk, grp in rd.groupby('site_key')}

    nbr_lookup = {}
    for focal, grp in dist_df.groupby('focal_key'):
        nbr_lookup[focal] = list(zip(grp['neighbor_key'], grp['distance_km']))

    records = []
    isolated = 0
    for _, spike in spikes.iterrows():
        focal_key  = spike['site_key']
        spike_date = spike['report_date']
        neighbors  = nbr_lookup.get(focal_key, [])
        if not neighbors:
            isolated += 1
            continue

        for nbr_key, dist_km in neighbors:
            if nbr_key not in rd_by_site:
                continue
            nbr_ts = rd_by_site[nbr_key].copy()
            nbr_ts['mfs'] = (
                (nbr_ts['report_date'].dt.year  - spike_date.year)  * 12 +
                (nbr_ts['report_date'].dt.month - spike_date.month)
            )
            window = nbr_ts[(nbr_ts['mfs'] >= -pre_months) & (nbr_ts['mfs'] <= post_months)]
            pre    = window[window['mfs'] < 0]
            if len(pre) < min_pre_obs:
                continue

            baseline_opex = pre['true_opex'].mean()
            baseline_ret  = pre['ret_wash_count'].mean()
            if pd.isna(baseline_opex) or baseline_opex <= 0:
                continue

            for _, row in window.iterrows():
                records.append({
                    'focal_key':        focal_key,
                    'spike_date':       spike_date,
                    'neighbor_key':     nbr_key,
                    'distance_km':      dist_km,
                    'months_from_spike': int(row['mfs']),
                    'opex_norm':        row['true_opex'] / baseline_opex,
                    'ret_wash_norm':    (row['ret_wash_count'] / max(baseline_ret, 1)
                                        if pd.notna(row['ret_wash_count']) else np.nan),
                })

    spike_resp_df = pd.DataFrame(records)
    if len(spike_resp_df) > 0:
        n_pairs = spike_resp_df.groupby(['focal_key', 'spike_date', 'neighbor_key']).ngroups
        print(f"Spike events with no neighbors:     {isolated}")
        print(f"Neighbor-spike pairs:               {n_pairs}")
    else:
        print("No neighbor-spike pairs found.")
    return spike_resp_df

spike_resp_df = analyze_incumbent_response_to_spikes(data, spikes, dist_df)


Spike events with no neighbors:     78
Neighbor-spike pairs:               410


In [ ]:
if len(spike_resp_df) == 0:
    print("No spike-incumbent pairs — cannot plot.")
else:
    INC_COLOR = '#E65100'

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'Neighbor OPEX (normalized) — do neighbors increase spend when a focal site runs a promotion?',
            'Neighbor Retail Wash Count (normalized) — do neighbors lose retail volume?',
        ],
        vertical_spacing=0.10,
    )

    for i, col in enumerate(['opex_norm', 'ret_wash_norm'], start=1):
        agg = (
            spike_resp_df.groupby('months_from_spike')[col]
            .agg(median='median',
                 q25=lambda s: s.quantile(0.25),
                 q75=lambda s: s.quantile(0.75),
                 n='count')
            .reset_index()
        )
        agg = agg[agg['n'] >= 5]
        x   = agg['months_from_spike']

        fig.add_trace(go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(agg['q75']) + list(agg['q25'][::-1]),
            fill='toself', fillcolor=INC_COLOR, opacity=0.12,
            line=dict(width=0), showlegend=False, hoverinfo='skip',
        ), row=i, col=1)
        fig.add_trace(go.Scatter(
            x=x, y=agg['median'],
            mode='lines+markers',
            line=dict(color=INC_COLOR, width=2.5), marker=dict(size=6),
            showlegend=False,
            hovertemplate='M%{x}<br>Median: %{y:.2f}× baseline<extra></extra>',
        ), row=i, col=1)

        fig.add_hline(y=1.0, line=dict(color='#333', dash='dash', width=1), row=i, col=1)
        fig.add_vline(x=0,   line=dict(color='#C62828', dash='dot', width=1.5), row=i, col=1)
        fig.update_yaxes(title_text='× baseline', tickformat='.2f', gridcolor='#EEE', row=i, col=1)
        fig.update_xaxes(showgrid=False, showline=True, linecolor='#CCC', row=i, col=1)

    n_pairs = spike_resp_df.groupby(['focal_key', 'spike_date', 'neighbor_key']).ngroups
    fig.update_xaxes(title_text='Months from OPEX spike', row=2, col=1)
    fig.update_layout(
        title=dict(
            text='<b>Neighbor Response to Focal Site Promotional Spike</b>'
                 f'<br><sub>{n_pairs} neighbor-spike pairs · '
                 'red dotted = spike month · dashed = pre-spike baseline · '
                 'shaded = 25–75th percentile</sub>',
            x=0.02, xanchor='left',
        ),
        plot_bgcolor='white',
        height=620,
        margin=dict(l=70, r=40, t=120, b=60),
    )
    fig.show()


## Section 6 — OPEX Investment Impact on Membership Growth

When a site spikes its OPEX by more than 20% above its own 6-month baseline — whether
through marketing spend, labour investments, or site improvements — a consistent pattern emerges:

- **Revenue lifts immediately** at the spike month (M=0)
- **Membership purchases accelerate** and compound over M+1 to M+6
- **Retail wash traffic converts** to membership: retail counts dip as customers switch plans
- **Active membership base expands** persistently for 6+ months post-spike

The event study below tracks these effects across 189 spike events,
normalized by each site's own pre-spike median (T=−6 to T=−1 = 1.0× baseline).
A Sankey chart shows the median membership flow from the spike month through M+3.

In [ ]:
monthly_membership = pd.read_csv(r"/Users/lakshyatomar/Desktop/OX/OX_v1/monthly_membership_activity.csv")
monthly_membership["bucket_date"] = pd.to_datetime(monthly_membership["bucket_date"])
monthly_membership["year"]  = monthly_membership["bucket_date"].dt.year
monthly_membership["month"] = monthly_membership["bucket_date"].dt.month

_mm = monthly_membership.drop_duplicates(subset=['location_name', 'month', 'year'])

# Columns to bring in from mm (exclude keys and columns already in data)
_mm_cols = ['location_name', 'month', 'year',
            'accounts_start', 'accounts_end', 'deactivated_accounts',
            'active_members', 'churn_rate_accounts', 'churn_rate_members', 'new_signups']

# data has ~568 duplicate (site_key, year, month) rows — some sites appear twice
# with conflicting expense values (one real row, one with 0).  Keep the row with
# the highest total_expenses so the event-study baseline is not corrupted.
_data_dedup = (
    data
    .sort_values('total_expenses', ascending=False)
    .drop_duplicates(subset=['site_key', 'year', 'month'], keep='first')
)

data_merged = _data_dedup.merge(
    _mm[_mm_cols],
    on=['location_name', 'month', 'year'],
    how='inner'
)

print(f'data rows:           {len(data):,}')
print(f'data rows (deduped): {len(_data_dedup):,}')
print(f'mm rows (deduped):   {len(_mm):,}')
print(f'merged rows:         {len(data_merged):,}')
print(f'dropped from data:   {len(_data_dedup) - len(data_merged):,} (sites missing in mm)')
# data_merged.head(3)


data rows:           2,679
data rows (deduped): 2,390
mm rows (deduped):   10,306
merged rows:         2,269
dropped from data:   121 (sites missing in mm)


In [ ]:
# Rows from data that had no match in monthly_membership
_mm_deduped = monthly_membership.drop_duplicates(subset=['location_name', 'month', 'year'])
data_unmatched = data.merge(
    _mm_deduped[['location_name', 'month', 'year']],
    on=['location_name', 'month', 'year'],
    how='left',
    indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

print(f'Unmatched rows: {len(data_unmatched):,}')
print(f'Unique sites:   {data_unmatched["location_name"].nunique()}')
print()
print(data_unmatched['location_name'].value_counts().to_string())
data_unmatched

Unmatched rows: 134
Unique sites:   12

location_name
Happy's Albany                  25
Happy's Vancouver               19
Vibe Car Wash Plano             19
Parrot's Car Wash Union City    17
Vibe Car Wash McKinney          16
Parrot's Car Wash Jackson       15
Happy's Medford                  5
Vibe Car Wash Arapaho            5
Vibe Car Wash Spring Valley      5
WashU Centennial                 5
WashU Fiesta                     2
Glacier Express 24th St          1


,client_name,client_id,site_id,location_name,year,quarter,month,report_date,mem_purchase_count,mem_wash_count,...,lon,active,timezone,location_id,source_schema,id,created_date,site_key,true_opex,region
2109,Glacier Express,glacierexpress_000240,3,Glacier Express 24th St,2024,4,12,2024-12-31,1195,2774,...,-108.577234,1,America/Denver,GE2,glacier_express,1243,2022-06-24 15:43:14,glacierexpress_000240__3,44018.46,West
2229,Happy Car Wash,happycarwash_000394,1,Happy's Albany,2022,4,12,2022-12-31,1087,2207,...,-123.112542,1,America/Los_Angeles,HCW1,happys_wash,1315,2021-11-09 23:35:46,happycarwash_000394__1,34183.82,West
2230,Happy Car Wash,happycarwash_000394,1,Happy's Albany,2023,1,1,2023-01-31,1065,2626,...,-123.112542,1,America/Los_Angeles,HCW1,happys_wash,1315,2021-11-09 23:35:46,happycarwash_000394__1,30938.48,West
2231,Happy Car Wash,happycarwash_000394,1,Happy's Albany,2023,1,2,2023-02-28,1072,2595,...,-123.112542,1,America/Los_Angeles,HCW1,happys_wash,1315,2021-11-09 23:35:46,happycarwash_000394__1,36086.13,West
2232,Happy Car Wash,happycarwash_000394,1,Happy's Albany,2023,1,3,2023-03-31,1091,2937,...,-123.112542,1,America/Los_Angeles,HCW1,happys_wash,1315,2021-11-09 23:35:46,happycarwash_000394__1,36369.75,West
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2656,Wash U,washu_000647,2,WashU Centennial,2024,2,5,2024-05-31,49,297,...,-115.144752,1,America/Los_Angeles,CENT,wash_u,2636,2024-04-26 12:11:02,washu_000647__2,60607.78,West
2657,Wash U,washu_000647,2,WashU Centennial,2024,2,6,2024-06-30,1884,4053,...,-115.144752,1,America/Los_Angeles,CENT,wash_u,2636,2024-04-26 12:11:02,washu_000647__2,67017.75,West
2658,Wash U,washu_000647,2,WashU Centennial,2024,3,7,2024-07-31,2061,5327,...,-115.144752,1,America/Los_Angeles,CENT,wash_u,2636,2024-04-26 12:11:02,washu_000647__2,63304.06,West
2659,Wash U,washu_000647,2,WashU Centennial,2024,3,8,2024-08-31,2472,5749,...,-115.144752,1,America/Los_Angeles,CENT,wash_u,2636,2024-04-26 12:11:02,washu_000647__2,65738.80,West


### Why the Join Has Unmatched Rows

The mismatch is **not** because a site is absent from `monthly_membership_activity.csv` entirely — it is because the **time ranges don't fully overlap**.

**Example:** `Glacier Express 24th St` (location_id `GE2`) has P&L rows in `data` for 2024, but `monthly_membership_activity.csv` only contains membership records up to a certain cut-off date for that site — so those 2024 rows find no matching counterpart and are dropped by the inner join.

**Root cause pattern:**
| File | What it contains |
| --- | --- |
| `merged_all.csv` (`data`) | Financial P&L rows — updated through the most recent reporting period |
| `monthly_membership_activity.csv` | Membership cohort activity — may lag behind or have a different extract date per operator |

As a result, a site like `Big Dan's Rome` or `Glacier Express 24th St` can have 2024 P&L data in `data` while `monthly_membership` only goes up to 2023 (or is missing altogether) for that operator.

**Implication for analysis:** `data_merged` (inner join result) only covers months where **both** the P&L record and the membership record exist. Any trend or KPI computed on `data_merged` excludes sites/months where membership data was not extracted — keep this in mind when interpreting portfolio-wide aggregates.

In [ ]:
def build_promo_membership_study(spikes, data_merged,
                                pre_months=6, post_months=12, min_pre_obs=3):
    """
    Event study: for each OPEX spike, track membership metrics at T=-6 to T=+12.
    Metrics: new_signups (true new accounts), mem_purchase_count (renewals), deactivated_accounts, active_members, churn_rate.
    Normalizes each metric by the site's own pre-spike median (baseline = 1.0).
    Returns promo_mm_df with one row per (spike, month_offset).
    """
    MM_COLS = [
        # From monthly_membership (churn/retention signal + true new signups)
        "deactivated_accounts", "active_members", "accounts_start", "accounts_end",
        "new_signups",
        # From P&L in data_merged (renewal volume + revenue signal)
        "mem_purchase_count", "ret_wash_count", "total_income",
    ]

    # Drop duplicate spikes (same site + same month detected twice due to raw-data duplication)
    spikes = spikes.drop_duplicates(subset=['site_key', 'report_date'])

    # Pre-compute: for each site_key, sorted deduplicated slice of data_merged
    dm = data_merged.copy()
    dm["report_date"] = pd.to_datetime(dm["report_date"])
    dm_by_site = {
        sk: (grp
             .sort_values("total_expenses", ascending=False)
             .drop_duplicates(subset=["year", "month"])
             .sort_values("report_date")
             .reset_index(drop=True))
        for sk, grp in dm.groupby("site_key")
    }

    records = []
    n_used, n_skipped = 0, 0

    for _, spike in spikes.iterrows():
        site_key   = spike["site_key"]
        spike_date = pd.to_datetime(spike["report_date"])

        if site_key not in dm_by_site:
            n_skipped += 1
            continue

        ts = dm_by_site[site_key].copy()
        ts["mfs"] = ((ts["report_date"].dt.year  - spike_date.year)  * 12 +
                     (ts["report_date"].dt.month - spike_date.month))

        window = ts[(ts["mfs"] >= -pre_months) & (ts["mfs"] <= post_months)]
        pre    = window[window["mfs"] < 0]

        if len(pre) < min_pre_obs:
            n_skipped += 1
            continue

        # Baseline: median of each metric over pre-spike window
        baselines = {}
        for c in MM_COLS:
            if c in pre.columns:
                bval = pre[c].median()
                baselines[c] = bval if (pd.notna(bval) and bval > 0) else np.nan

        n_used += 1

        for _, row in window.iterrows():
            rec = {
                "site_key":          site_key,
                "spike_date":        spike_date,
                "months_from_spike": int(row["mfs"]),
            }
            for c in MM_COLS:
                rec[c] = row[c] if c in row.index else np.nan
                bl = baselines.get(c)
                rec[f"{c}_norm"] = (row[c] / bl) if (pd.notna(bl) and pd.notna(row.get(c, np.nan))) else np.nan
            # Carry churn rates as-is (already in % — not normalized)
            rec["churn_rate_accounts"] = row.get("churn_rate_accounts", np.nan)
            rec["churn_rate_members"]  = row.get("churn_rate_members",  np.nan)
            records.append(rec)

    promo_mm_df = pd.DataFrame(records)

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"Spikes with membership data : {n_used:>4}  (skipped {n_skipped} — no data or <{min_pre_obs} pre-months)")
    if len(promo_mm_df) > 0:
        n_sites = promo_mm_df["site_key"].nunique()
        print(f"Sites covered               : {n_sites}")
        print()

        for t_ref, label in [(0, "M=0 (spike month)"), (1, "M+1"), (2, "M+2"), (3, "M+3")]:
            sub = promo_mm_df[promo_mm_df["months_from_spike"] == t_ref]
            if len(sub) == 0:
                continue
            pre_sub = promo_mm_df[promo_mm_df["months_from_spike"] < 0]
            print(f"  {label}:")
            for c in ["mem_purchase_count", "deactivated_accounts", "active_members"]:
                med_now  = sub[c].median()
                med_base = pre_sub[c].median()
                norm     = sub[f"{c}_norm"].median()
                if pd.notna(med_now):
                    print(f"    {c:<26}: {med_now:>8.0f}  (baseline {med_base:.0f},  ratio {norm:.3f}×)")
        print()
    return promo_mm_df

promo_mm_df = build_promo_membership_study(spikes, data_merged)
promo_mm_df.head()


Spikes with membership data :  189  (skipped 46 — no data or <3 pre-months)
Sites covered               : 79

  M=0 (spike month):
    mem_purchase_count        :     2666  (baseline 2410,  ratio 1.098×)
    deactivated_accounts      :      237  (baseline 220,  ratio 1.083×)
    active_members            :     2893  (baseline 2652,  ratio 1.089×)
  M+1:
    mem_purchase_count        :     2699  (baseline 2410,  ratio 1.105×)
    deactivated_accounts      :      246  (baseline 220,  ratio 1.074×)
    active_members            :     2909  (baseline 2652,  ratio 1.104×)
  M+2:
    mem_purchase_count        :     2824  (baseline 2410,  ratio 1.117×)
    deactivated_accounts      :      237  (baseline 220,  ratio 1.054×)
    active_members            :     2957  (baseline 2652,  ratio 1.120×)
  M+3:
    mem_purchase_count        :     2894  (baseline 2410,  ratio 1.140×)
    deactivated_accounts      :      240  (baseline 220,  ratio 1.064×)
    active_members            :     3055  (baseli

,site_key,spike_date,months_from_spike,deactivated_accounts,deactivated_accounts_norm,active_members,active_members_norm,accounts_start,accounts_start_norm,accounts_end,...,new_signups,new_signups_norm,mem_purchase_count,mem_purchase_count_norm,ret_wash_count,ret_wash_count_norm,total_income,total_income_norm,churn_rate_accounts,churn_rate_members
0,bluewave_000567__1,2024-05-31,-6,3175,10.143770,4032,1.256075,6956,2.263954,3931,...,150,0.597610,1924,0.893222,1591,0.864674,77776.77,0.907900,45.644048,44.958819
1,bluewave_000567__1,2024-05-31,-5,1112,3.552716,3151,0.981620,3931,1.279414,3036,...,217,0.864542,2000,0.928505,1542,0.838043,76002.01,0.887183,28.287967,27.380952
2,bluewave_000567__1,2024-05-31,-4,272,0.869010,3116,0.970717,3036,0.988120,2994,...,230,0.916335,2092,0.971216,1620,0.880435,78648.34,0.918074,8.959157,8.568708
3,bluewave_000567__1,2024-05-31,-3,295,0.942492,3099,0.965421,2994,0.974451,2971,...,272,1.083665,2216,1.028784,2060,1.119565,92684.98,1.081926,9.853039,9.338896
4,bluewave_000567__1,2024-05-31,-2,312,0.996805,3269,1.018380,2971,0.966965,3109,...,450,1.792829,2498,1.159703,2554,1.388043,104529.26,1.220186,10.501515,9.874153


In [ ]:
if len(promo_mm_df) == 0:
    print("No membership data for event study.")
else:
    # (col, label, color, is_pct) — is_pct=True → show as % not ×baseline
    PANELS = [
        ("total_income_norm",         "Revenue",                    "#1B5E20", False),
        ("mem_purchase_count_norm",   "Membership Purchases (P&L)", "#1565C0", False),
        ("active_members_norm",       "Active Members",             "#2E7D32", False),
        ("ret_wash_count_norm",       "Retail Wash Count",          "#E65100", False),
        ("deactivated_accounts_norm", "Deactivated Accounts",       "#C62828", False),
        ("churn_rate_accounts",       "Churn Rate (Accounts)",      "#AD1457", True),
        ("churn_rate_members",        "Churn Rate (Members)",       "#FF6F00", True),
    ]

    fig = make_subplots(
        rows=7, cols=1, shared_xaxes=True,
        subplot_titles=[p[1] for p in PANELS],
        vertical_spacing=0.06,
    )

    n_spikes = promo_mm_df.groupby(["site_key","spike_date"]).ngroups

    for row_idx, (col, label, color, is_pct) in enumerate(PANELS, start=1):
        agg = (promo_mm_df.dropna(subset=[col])
               .groupby("months_from_spike")[col]
               .agg(median="median",
                    q25=lambda s: s.quantile(0.25),
                    q75=lambda s: s.quantile(0.75),
                    n="count")
               .reset_index())
        agg = agg[agg["n"] >= 5]
        x   = agg["months_from_spike"]

        # IQR band
        fig.add_trace(go.Scatter(
            x=list(x) + list(x[::-1]),
            y=list(agg["q75"]) + list(agg["q25"][::-1]),
            fill="toself", fillcolor=color, opacity=0.12,
            line=dict(width=0), showlegend=False, hoverinfo="skip",
        ), row=row_idx, col=1)

        # Median line
        fig.add_trace(go.Scatter(
            x=x, y=agg["median"],
            mode="lines+markers",
            name=label,
            line=dict(color=color, width=2.5),
            marker=dict(size=5, color=color),
            hovertemplate=f"M%{{x}}: %{{y:.2f}}" + ("%" if is_pct else "×") + f"<extra>{label}</extra>",
        ), row=row_idx, col=1)

        # Baseline reference (not meaningful for % panels)
        if not is_pct:
            fig.add_hline(y=1.0, line=dict(color="#999", dash="dash", width=1),
                          row=row_idx, col=1)
        # Spike month marker
        fig.add_vline(x=0, line=dict(color="#333", dash="dot", width=1.5),
                      row=row_idx, col=1)

        y_suf = "%" if is_pct else "×"
        fig.update_yaxes(title_text=label, ticksuffix=y_suf,
                         gridcolor="#EEE", row=row_idx, col=1)
        fig.update_xaxes(showgrid=False, showline=True,
                         linecolor="#CCC", row=row_idx, col=1)

    fig.update_xaxes(
        title_text="Months from OPEX Spike  (0 = investment month)",
        tickvals=list(range(-6, 13)),
        row=7, col=1,
    )
    fig.update_layout(
        title=dict(
            text=(
                f"<b>OPEX Investment Impact on Membership Growth</b>"
                f"<br><sub>{n_spikes} spike events · "
                "normalized by each site's pre-spike median (1.0 = baseline) · "
                "shaded = 25–75th percentile · dotted line = investment month</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        showlegend=False,
        plot_bgcolor="white",
        height=1600,
        margin=dict(l=100, r=40, t=150, b=70),
    )
    fig.show()


In [ ]:
if len(promo_mm_df) > 0:
    _m0 = promo_mm_df[promo_mm_df["months_from_spike"] == 0]

    med_new      = max(_m0["new_signups"].median(), 1)        # true new account sign-ups
    med_existing = max(_m0["accounts_start"].median(), 1)     # existing members entering M=0

    # Per-month churn rates M+1 to M+6 from observed event study data
    cr = {}
    for t in range(1, 7):
        sub_t = promo_mm_df[promo_mm_df["months_from_spike"] == t]
        cr[t] = (sub_t["churn_rate_accounts"].median() or 0) / 100

    # Cohort survival: apply monthly churn to the original M=0 pool each month
    pool   = med_new + med_existing
    active = {}
    deact  = {}
    active[1] = pool * (1 - cr[1])
    deact[1]  = pool * cr[1]
    for t in range(2, 7):
        active[t] = active[t - 1] * (1 - cr[t])
        deact[t]  = active[t - 1] * cr[t]

    n_spikes = promo_mm_df.groupby(["site_key", "spike_date"]).ngroups

    # ── Nodes ─────────────────────────────────────────────────────────────────
    node_labels = [
        f"OPEX Spike<br>(n={n_spikes})",
        f"New Sign-ups<br>~{med_new:,.0f} / site",
        f"Existing Members<br>~{med_existing:,.0f} / site",
    ]
    node_hover = [
        f"OPEX Spike · {n_spikes} events",
        f"True new account sign-ups at M=0<br>~{med_new:,.0f} accounts / site<br>(source: new_signups, monthly_membership)",
        f"Existing members at start of spike month<br>~{med_existing:,.0f} accounts / site<br>(source: accounts_start, monthly_membership)",
    ]
    for t in range(1, 7):
        pct = active[t] / pool * 100
        node_labels.append(f"M+{t}<br>{pct:.0f}% of pool")
        node_hover.append(
            f"Active at M+{t}<br>"
            f"~{active[t]:,.0f} accounts remaining<br>"
            f"{pct:.0f}% of starting cohort ({pool:,.0f})<br>"
            f"Monthly churn: {cr[t]*100:.2f}%"
        )
        node_labels.append(f"Deact M+{t}")
        node_hover.append(
            f"Deactivated during M+{t}<br>"
            f"~{deact[t]:,.0f} accounts<br>"
            f"Churn rate: {cr[t]*100:.2f}%"
        )

    node_color = ["#78909C", "#1565C0", "#2E7D32"] + ["#1B5E20", "#B71C1C"] * 6
    node_x = [
        0.01,  0.13,  0.13,
        0.27,  0.27,  0.41,  0.41,
        0.55,  0.55,  0.68,  0.68,
        0.81,  0.81,  0.97,  0.97,
    ]
    node_y = [
        0.50,  0.22,  0.75,
        0.28,  0.88,  0.28,  0.88,
        0.28,  0.88,  0.28,  0.88,
        0.28,  0.88,  0.28,  0.88,
    ]

    # ── Links ─────────────────────────────────────────────────────────────────
    link_src = [
        0, 0,
        1, 1, 2, 2,
        3,  3,
        5,  5,
        7,  7,
        9,  9,
        11, 11,
    ]
    link_tgt = [
        1, 2,
        3, 4, 3, 4,
        5,  6,
        7,  8,
        9,  10,
        11, 12,
        13, 14,
    ]
    link_val = [
        med_new, med_existing,
        med_new * (1 - cr[1]), med_new * cr[1],
        med_existing * (1 - cr[1]), med_existing * cr[1],
        active[1] * (1 - cr[2]), active[1] * cr[2],
        active[2] * (1 - cr[3]), active[2] * cr[3],
        active[3] * (1 - cr[4]), active[3] * cr[4],
        active[4] * (1 - cr[5]), active[4] * cr[5],
        active[5] * (1 - cr[6]), active[5] * cr[6],
    ]
    green_active = "rgba(27,94,32,0.35)"
    red_deact    = "rgba(183,28,28,0.35)"
    link_color = [
        "rgba(21,101,192,0.30)", "rgba(46,125,50,0.25)",
        "rgba(21,101,192,0.25)", red_deact,
        "rgba(46,125,50,0.20)",  red_deact,
        green_active, red_deact,
        green_active, red_deact,
        green_active, red_deact,
        green_active, red_deact,
        green_active, red_deact,
    ]

    fig = go.Figure(go.Sankey(
        arrangement="snap",
        node=dict(
            pad=10, thickness=18,
            label=node_labels,
            customdata=node_hover,
            hovertemplate="%{customdata}<extra></extra>",
            color=node_color,
            x=node_x, y=node_y,
        ),
        link=dict(
            source=link_src, target=link_tgt,
            value=link_val, color=link_color,
        ),
    ))
    fig.update_layout(
        title=dict(
            text=(
                f"<b>Membership Cohort Survival: OPEX Investment Month → M+6</b>"
                f"<br><sub>Starting cohort: {pool:,.0f} accounts / site at M=0 "
                f"({med_new:,.0f} new sign-ups + {med_existing:,.0f} existing) · "
                f"{active[6]:,.0f} ({active[6]/pool*100:.0f}%) still active at M+6 · "
                f"per-month churn from observed median rates · {n_spikes} spike events</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        font_size=11,
        height=680,
        margin=dict(l=40, r=40, t=150, b=40),
    )
    fig.show()


In [ ]:
# if len(promo_mm_df) > 0:
#     # ── Inputs ───────────────────────────────────────────────────────────────
#     _m_neg6 = promo_mm_df[promo_mm_df["months_from_spike"] == -6]
#     _m0     = promo_mm_df[promo_mm_df["months_from_spike"] ==  0]

#     start_pool = max(_m_neg6["accounts_start"].median(), 1)   # membership base 6 months before spike
#     med_new    = max(_m0["new_signups"].median(),  1)          # true new account sign-ups in spike month

#     # Per-month churn rates M-6 to M+6
#     cr = {}
#     for t in range(-6, 7):
#         sub_t = promo_mm_df[promo_mm_df["months_from_spike"] == t]
#         cr[t] = (sub_t["churn_rate_accounts"].median() or 0) / 100

#     # ── Pre-spike cohort survival (M-6 → M-1) ────────────────────────────────
#     pre_active = {-6: start_pool}
#     pre_deact  = {}
#     for t in range(-6, 0):
#         pre_deact[t] = pre_active[t] * cr[t]
#         if t < -1:
#             pre_active[t + 1] = pre_active[t] * (1 - cr[t])
#     existing_m0 = pre_active[-1] * (1 - cr[-1])   # pre-spike survivors entering M=0

#     # ── Post-spike cohort survival (M=0 combined pool → M+6) ─────────────────
#     combined   = existing_m0 + med_new
#     post_active = {0: combined}
#     post_deact  = {}
#     for t in range(1, 7):
#         post_deact[t]  = post_active[t - 1] * cr[t]
#         post_active[t] = post_active[t - 1] * (1 - cr[t])

#     n_spikes = promo_mm_df.groupby(["site_key", "spike_date"]).ngroups

#     # ── Node layout ───────────────────────────────────────────────────────────
#     # 0  = Pool M-6
#     # 1,2  = Active M-5, Deact M-6
#     # 3,4  = Active M-4, Deact M-5
#     # 5,6  = Active M-3, Deact M-4
#     # 7,8  = Active M-2, Deact M-3
#     # 9,10 = Active M-1, Deact M-2
#     # 11,12 = Existing M=0, Deact M-1
#     # 13,14 = OPEX Spike, New Sign-ups M=0
#     # 15,16 = Active M+1, Deact M+1   ...   25,26 = Active M+6, Deact M+6

#     pct6 = post_active[6] / combined * 100

#     node_labels = [
#         f"Membership Pool M−6<br>{start_pool:,.0f} accounts / site",  # 0
#         f"Active M−5<br>{pre_active[-5]:,.0f}",   # 1
#         f"Deact M−6 ▼{pre_deact[-6]:,.0f}",       # 2
#         f"Active M−4<br>{pre_active[-4]:,.0f}",   # 3
#         f"Deact M−5 ▼{pre_deact[-5]:,.0f}",       # 4
#         f"Active M−3<br>{pre_active[-3]:,.0f}",   # 5
#         f"Deact M−4 ▼{pre_deact[-4]:,.0f}",       # 6
#         f"Active M−2<br>{pre_active[-2]:,.0f}",   # 7
#         f"Deact M−3 ▼{pre_deact[-3]:,.0f}",       # 8
#         f"Active M−1<br>{pre_active[-1]:,.0f}",   # 9
#         f"Deact M−2 ▼{pre_deact[-2]:,.0f}",       # 10
#         f"Pre-spike<br>Survivors<br>{existing_m0:,.0f}",  # 11
#         f"Deact M−1 ▼{pre_deact[-1]:,.0f}",       # 12
#         f"OPEX Spike<br>(n={n_spikes})",           # 13
#         f"New Sign-ups<br>M=0: {med_new:,.0f}",    # 14
#     ]
#     for t in range(1, 7):
#         pct = post_active[t] / combined * 100
#         node_labels.append(f"Active M+{t}<br>{post_active[t]:,.0f} ({pct:.0f}%)")  # 15,17,19,21,23,25
#         node_labels.append(f"Deact M+{t} ▼{post_deact[t]:,.0f}")                  # 16,18,20,22,24,26

#     GREEN = "#1B5E20"; RED = "#B71C1C"; GREY = "#78909C"
#     BLUE  = "#1565C0"; SLATE = "#546E7A"; OLIVE = "#2E7D32"

#     node_color = (
#         [SLATE]                          # 0  Pool M-6
#         + [GREEN, RED] * 5               # 1-10  Active M-5…M-1 / Deact M-6…M-2
#         + [OLIVE, RED, GREY, BLUE]       # 11-14 Existing, Deact M-1, Spike, New
#         + [GREEN, RED] * 6               # 15-26 Active/Deact M+1…M+6
#     )
#     node_x = (
#         [0.01]
#         + [x for v in [0.09, 0.17, 0.25, 0.33, 0.41] for x in [v, v]]   # 1-10
#         + [0.50, 0.50, 0.50, 0.50]                                         # 11-14
#         + [x for v in [0.59, 0.67, 0.75, 0.83, 0.91, 0.99] for x in [v, v]]  # 15-26
#     )
#     node_y = (
#         [0.35]
#         + [y for _ in range(5) for y in [0.35, 0.90]]   # 1-10  active top, deact bottom
#         + [0.58, 0.97, 0.06, 0.20]                        # 11 Existing, 12 Deact M-1, 13 Spike, 14 New
#         + [y for _ in range(6) for y in [0.35, 0.90]]    # 15-26
#     )

#     # ── Links ────────────────────────────────────────────────────────────────
#     G  = "rgba(27,94,32,0.30)"   # pre-spike active flow
#     R  = "rgba(183,28,28,0.32)"  # deact
#     B  = "rgba(21,101,192,0.38)" # OPEX → new purchases
#     GP = "rgba(27,94,32,0.38)"   # post-spike active flow
#     Bx = "rgba(21,101,192,0.28)" # new purchases → active
#     Ox = "rgba(46,125,50,0.28)"  # existing → active post

#     active_nodes = [1, 3, 5, 7, 9]   # Active M-5 … M-1
#     deact_nodes  = [2, 4, 6, 8, 10]  # Deact M-6 … M-2

#     link_src, link_tgt, link_val, link_col = [], [], [], []

#     # Pre-spike chain: Pool M-6 → ...
#     src = 0
#     for i, t in enumerate(range(-6, -1)):   # t = -6,-5,-4,-3,-2
#         nxt_active = active_nodes[i]        # Active at t+1 (node 1,3,5,7,9)
#         nxt_deact  = deact_nodes[i]         # Deact at t   (node 2,4,6,8,10)
#         link_src += [src, src];   link_tgt += [nxt_active, nxt_deact]
#         link_val += [pre_active[t] * (1 - cr[t]), pre_deact[t]]
#         link_col += [G, R]
#         src = nxt_active
#     # Active M-1 (node 9) → Existing M=0 (11) + Deact M-1 (12)
#     link_src += [9, 9];   link_tgt += [11, 12]
#     link_val += [existing_m0, pre_deact[-1]]
#     link_col += [G, R]

#     # OPEX Spike (13) → New Purchases (14)
#     link_src += [13];   link_tgt += [14]
#     link_val += [med_new];   link_col += [B]

#     # Both existing (11) + new purchases (14) → Active M+1 (15) + Deact M+1 (16)
#     link_src += [11, 14, 11, 14]
#     link_tgt += [15,  15, 16,  16]
#     link_val += [existing_m0 * (1 - cr[1]), med_new * (1 - cr[1]),
#                  existing_m0 * cr[1],        med_new * cr[1]]
#     link_col += [Ox, Bx, R, R]

#     # Post-spike chain: Active M+1 → M+2 → … → M+6
#     post_act_nodes  = [15, 17, 19, 21, 23, 25]
#     post_deac_nodes = [16, 18, 20, 22, 24, 26]
#     for i in range(1, 6):   # M+1 → M+2 through M+5 → M+6
#         t = i + 1
#         link_src += [post_act_nodes[i - 1], post_act_nodes[i - 1]]
#         link_tgt += [post_act_nodes[i],     post_deac_nodes[i]]
#         link_val += [post_active[i] * (1 - cr[t]), post_active[i] * cr[t]]
#         link_col += [GP, R]

#     # ── Chart ─────────────────────────────────────────────────────────────────
#     fig = go.Figure(go.Sankey(
#         arrangement="snap",
#         node=dict(pad=8, thickness=16, label=node_labels,
#                   color=node_color, x=node_x, y=node_y),
#         link=dict(source=link_src, target=link_tgt,
#                   value=link_val,  color=link_col),
#     ))
#     fig.update_layout(
#         title=dict(
#             text=(
#                 "<b>Full Membership Journey: M−6 → OPEX Spike → M+6</b>"
#                 f"<br><sub>Pre-spike pool: {start_pool:,.0f} accounts → "
#                 f"{existing_m0:,.0f} survive to M=0 · "
#                 f"OPEX spike adds {med_new:,.0f} new sign-ups · "
#                 f"combined pool: {combined:,.0f} → "
#                 f"{post_active[6]:,.0f} ({pct6:.0f}%) active at M+6 · "
#                 f"churn ~8–9%/month throughout</sub>"
#             ),
#             x=0.02, xanchor="left",
#         ),
#         font_size=10,
#         height=700,
#         margin=dict(l=30, r=30, t=160, b=30),
#     )
#     fig.show()


In [ ]:
if len(promo_mm_df) > 0:
    from plotly.subplots import make_subplots

    months  = list(range(-6, 7))
    mlabels = {t: "M=0" if t == 0 else f"M{t:+d}" for t in months}

    # Colour scheme: grey pre-spike, amber M=0 OPEX, green post-spike
    line_col = {t: "#E65100"  if t == 0 else ("#78909C" if t < 0 else "#1B5E20") for t in months}
    fill_col = {
        t: "rgba(230,81,0,0.22)"     if t == 0
        else "rgba(120,144,156,0.20)" if t < 0
        else "rgba(27,94,32,0.20)"
        for t in months
    }

    METRICS = [
        ("new_signups",          "New Sign-ups per Site",
         "True new membership accounts opened each month"),
        ("accounts_start",       "Membership Pool per Site",
         "Total active accounts entering each month (accounts_start)"),
        ("deactivated_accounts", "Deactivations per Site",
         "Membership accounts cancelled each month"),
    ]

    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.10,
        subplot_titles=[m[1] for m in METRICS],
    )

    n_spikes = promo_mm_df.groupby(["site_key", "spike_date"]).ngroups

    for r, (col, _, desc) in enumerate(METRICS, 1):
        for t in months:
            vals = promo_mm_df.loc[
                promo_mm_df["months_from_spike"] == t, col
            ].dropna()
            med  = vals.median()
            n    = len(vals)
            fig.add_trace(
                go.Violin(
                    y=vals,
                    x=[mlabels[t]] * n,
                    name=mlabels[t],
                    box=dict(visible=True),
                    meanline=dict(visible=True, color="white", width=1.5),
                    line_color=line_col[t],
                    fillcolor=fill_col[t],
                    opacity=0.90,
                    points="outliers",
                    marker=dict(size=2.5, opacity=0.30, color=line_col[t]),
                    customdata=[[f"{mlabels[t]}  ·  {desc}<br>"
                                 f"n={n} events  ·  median={med:,.0f}  ·  "
                                 f"IQR [{vals.quantile(0.25):,.0f} – {vals.quantile(0.75):,.0f}]"]] * n,
                    hovertemplate="%{customdata[0]}<extra></extra>",
                    showlegend=False,
                ),
                row=r, col=1,
            )

        # Bracket annotations: "Pre-OPEX", "OPEX Spike", "Post-OPEX"
        # (added only once, on row 1, using paper y-reference)
        if r == 1:
            for txt, xpos, clr in [
                ("Pre-OPEX baseline",  "M-5",  "#78909C"),
                ("OPEX Spike ▲",       "M=0",  "#E65100"),
                ("Post-OPEX growth",   "M+4",  "#1B5E20"),
            ]:
                fig.add_annotation(
                    x=xpos, y=1.045, xref="x", yref="paper",
                    text=f"<b>{txt}</b>",
                    showarrow=False,
                    font=dict(size=9, color=clr),
                    align="center",
                )

    # Shared x-axis: category order, tick labels, bottom title
    fig.update_xaxes(
        categoryorder="array",
        categoryarray=[mlabels[t] for t in months],
        tickangle=0,
        row=3, col=1,
        title_text="Month relative to OPEX spike",
    )
    # Mirror categories on rows 1 & 2 (shared axis)
    for r in [1, 2]:
        fig.update_xaxes(
            categoryorder="array",
            categoryarray=[mlabels[t] for t in months],
            row=r, col=1,
        )

    # Y-axis labels
    fig.update_yaxes(title_text="Accounts", title_standoff=6, row=1, col=1)
    fig.update_yaxes(title_text="Accounts", title_standoff=6, row=2, col=1)
    fig.update_yaxes(title_text="Accounts", title_standoff=6, row=3, col=1)

    fig.update_layout(
        title=dict(
            text=(
                "<b>Membership Metric Distributions by Month: M-6 → OPEX Spike (M=0) → M+6</b>"
                f"<br><sub>Across {n_spikes} OPEX spike events  ·  "
                "violin = full distribution  ·  box = IQR  ·  white line = median  ·  "
                "dots = outliers  ·  "
                "<span style='color:#78909C'>grey = pre-spike</span>  ·  "
                "<span style='color:#E65100'>orange = M=0 OPEX</span>  ·  "
                "<span style='color:#1B5E20'>green = post-spike</span>"
                "</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        height=900,
        font_size=10,
        margin=dict(l=70, r=20, t=140, b=60),
        plot_bgcolor="rgba(250,250,250,1)",
        paper_bgcolor="white",
    )

    # Light horizontal gridlines only
    for r in [1, 2, 3]:
        fig.update_yaxes(
            showgrid=True, gridcolor="rgba(200,200,200,0.4)",
            gridwidth=0.5, zeroline=False,
            row=r, col=1,
        )
        fig.update_xaxes(showgrid=False, row=r, col=1)

    fig.show()


## Section 7 — Churn Seasonality & Membership Flow

Three consistent seasonal patterns emerge from the data:

| Window | Churn | Net Flow | Story |
|---|---|---|---|
| **Jan – Feb** | Low (8.1–8.0%) | +32 to +96 | Post-holiday re-commit; stable base |
| **Mar** | Moderate (9.7%) | **+125 — peak growth** | Spring sign-up surge overwhelms churn |
| **Apr – May** | High (9.6–9.9%) | **May = −19 — only negative month** | Churn spike + weakest sign-ups; net-membership shrinks at ~58% of sites |
| **Jul – Aug** | High (9.7–10.2%) | +29 to +56 | High churn but summer sign-ups recover it |
| **Oct** | High (9.9%) | **+94 — second growth peak** | Fall re-engagement drives strong acquisition |
| **Nov – Dec** | Low (8.1–8.2%) | Flat | Year-end stability; lower acquisition but low churn keeps base intact |

**Operational implications:**  
- **Protect May**: lowest net-growth month; targeted retention campaigns or pause-plan offers may prevent net decline  
- **Load OPEX in Feb–Mar**: the natural sign-up surge means investment ROI compounds on a larger pool  
- **Expect Aug churn spike**: plan for it rather than react — it has always recovered by Sep


In [ ]:
# if "data_merged" in dir():
#     from plotly.subplots import make_subplots
#     import calendar

#     MONTH_NAMES = [calendar.month_abbr[m] for m in range(1, 13)]

#     dm_s = data_merged.dropna(
#         subset=["churn_rate_accounts", "new_signups", "deactivated_accounts"]
#     ).copy()
#     # Filter extreme churn outliers (data artefacts) for visualisation
#     dm_s = dm_s[dm_s["churn_rate_accounts"].between(0, 50)]
#     dm_s["net_flow"]   = dm_s["new_signups"] - dm_s["deactivated_accounts"]
#     dm_s["month_name"] = dm_s["month"].map(lambda m: MONTH_NAMES[m - 1])

#     # Median net flow per month — determines bar colour in panel 3
#     med_flow = dm_s.groupby("month")["net_flow"].median()
#     net_color = {
#         m: "rgba(27,94,32,0.22)" if med_flow[m] > 0 else "rgba(183,28,28,0.22)"
#         for m in range(1, 13)
#     }

#     # Churn colour: blue-cool for low-churn months, warm-red for high-churn
#     med_churn = dm_s.groupby("month")["churn_rate_accounts"].median()
#     lo, hi = med_churn.min(), med_churn.max()
#     def churn_color(m, alpha=0.22):
#         t = (med_churn[m] - lo) / (hi - lo)   # 0 = coolest, 1 = hottest
#         r = int(30  + t * (183 - 30))
#         g = int(136 + t * (28  - 136))
#         b = int(229 + t * (28  - 229))
#         return f"rgba({r},{g},{b},{alpha})"

#     # Sign-up months: use a consistent teal
#     TEAL = "rgba(0,121,107,0.22)"

#     METRICS = [
#         ("churn_rate_accounts", "Churn Rate (%)",          "churn"),
#         ("new_signups",         "New Sign-ups / Site",     "signups"),
#         ("net_flow",            "Net Membership Flow\n(Sign-ups − Deactivations)", "netflow"),
#     ]

#     fig = make_subplots(
#         rows=3, cols=1,
#         shared_xaxes=True,
#         vertical_spacing=0.09,
#         subplot_titles=[m[1] for m in METRICS],
#     )

#     n_sites = dm_s["site_key"].nunique()
#     n_rows  = len(dm_s)

#     for r, (col, _, kind) in enumerate(METRICS, 1):
#         for m in range(1, 13):
#             vals = dm_s.loc[dm_s["month"] == m, col].dropna()
#             med  = vals.median()
#             pct_pos = (vals > 0).mean() * 100 if kind == "netflow" else None

#             if kind == "churn":
#                 lc = churn_color(m, alpha=0.9)
#                 fc = churn_color(m, alpha=0.22)
#             elif kind == "signups":
#                 lc, fc = "#00796B", TEAL
#             else:   # net flow
#                 lc = "#1B5E20" if med > 0 else "#B71C1C"
#                 fc = net_color[m]

#             hover = (
#                 f"<b>{MONTH_NAMES[m-1]}</b><br>"
#                 f"Median: {med:+.1f}{'%' if kind == 'churn' else ''}<br>"
#                 f"IQR: [{vals.quantile(0.25):+.1f} – {vals.quantile(0.75):+.1f}]<br>"
#                 f"n = {len(vals)} site-months"
#                 + (f"<br>Net-positive sites: {pct_pos:.0f}%" if pct_pos is not None else "")
#             )

#             fig.add_trace(
#                 go.Violin(
#                     y=vals,
#                     x=[MONTH_NAMES[m - 1]] * len(vals),
#                     name=MONTH_NAMES[m - 1],
#                     box=dict(visible=True),
#                     meanline=dict(visible=True, color="white", width=1.5),
#                     line_color=lc,
#                     fillcolor=fc,
#                     opacity=0.90,
#                     points="outliers",
#                     marker=dict(size=2.5, opacity=0.25, color=lc),
#                     customdata=[[hover]] * len(vals),
#                     hovertemplate="%{customdata[0]}<extra></extra>",
#                     showlegend=False,
#                 ),
#                 row=r, col=1,
#             )

#         # Zero reference line for net-flow panel
#         if kind == "netflow":
#             fig.add_hline(y=0, line_color="rgba(0,0,0,0.35)",
#                           line_dash="dot", line_width=1.2, row=r, col=1)

#     # Annotations: bracket labels on row 1
#     BRACKETS = [
#         ("Low churn\nwinter",   "Jan",  "#1565C0"),
#         ("Spring surge",        "Mar",  "#00796B"),
#         ("May danger zone",     "May",  "#B71C1C"),
#         ("Fall re-engagement",  "Oct",  "#E65100"),
#         ("Year-end stability",  "Dec",  "#546E7A"),
#     ]
#     for txt, xpos, clr in BRACKETS:
#         fig.add_annotation(
#             x=xpos, y=1.055, xref="x", yref="paper",
#             text=f"<b>{txt}</b>",
#             showarrow=False, font=dict(size=8, color=clr), align="center",
#         )

#     # Axis formatting
#     fig.update_xaxes(
#         categoryorder="array",
#         categoryarray=MONTH_NAMES,
#         tickangle=0,
#     )
#     fig.update_xaxes(title_text="Calendar month", row=3, col=1)
#     fig.update_yaxes(title_text="Churn %",  title_standoff=4, row=1, col=1)
#     fig.update_yaxes(title_text="Accounts", title_standoff=4, row=2, col=1)
#     fig.update_yaxes(title_text="Accounts", title_standoff=4, row=3, col=1)

#     for r in range(1, 4):
#         fig.update_yaxes(
#             showgrid=True, gridcolor="rgba(200,200,200,0.4)",
#             gridwidth=0.5, zeroline=False, row=r, col=1,
#         )
#         fig.update_xaxes(showgrid=False, row=r, col=1)

#     fig.update_layout(
#         title=dict(
#             text=(
#                 "<b>Churn Seasonality & Membership Flow — All Sites, All Time</b>"
#                 f"<br><sub>{n_sites} sites · {n_rows:,} site-months · "
#                 "churn violin colour: blue = low churn month, red-warm = high churn month · "
#                 "net flow: green = net-positive median, red = net-negative</sub>"
#             ),
#             x=0.02, xanchor="left",
#         ),
#         height=920,
#         font_size=10,
#         margin=dict(l=70, r=20, t=140, b=60),
#         plot_bgcolor="rgba(250,250,250,1)",
#         paper_bgcolor="white",
#     )
#     fig.show()
# else:
#     print("data_merged not found — run cell 51 first")


## Section 9 — Classifying OPEX Spikes: Customer-Facing vs Operational

Not every OPEX spike represents the same kind of investment. We hypothesise two distinct types:

| Type | OPEX Goes To | Expected Signal |
|---|---|---|
| **Customer-Facing** | Marketing, promotions, site experience improvements | New sign-ups ↑, Active members ↑, Membership purchases ↑, Revenue ↑ |
| **Operational** | Equipment, machinery, labour, maintenance | Acquisition flat or down — the site is investing in capacity, not customers |

**Classification method**: For each spike event, compute the average of `new_signups_norm` and `active_members_norm` at M+1, M+2, M+3 relative to that site's own pre-spike baseline. A spike is **Customer-Facing** if this combined post-spike signal is above the cross-spike median; **Operational** otherwise.

The two groups produce starkly different patterns across all downstream metrics — including retail wash count, which moves in *opposite directions* for the two types.


In [ ]:
if "promo_mm_df" not in dir():
    print("Run cell 56 (build_promo_membership_study) first.")
else:
    from plotly.subplots import make_subplots

    # ── Step 1: Classify each spike ───────────────────────────────────────────
    post_data = promo_mm_df[promo_mm_df["months_from_spike"].between(1, 3)]

    spike_agg = (
        post_data
        .groupby(["site_key", "spike_date"])
        .agg(
            new_sig_norm=("new_signups_norm",        "mean"),
            active_norm     =("active_members_norm", "mean"),
            active_acct_norm=("accounts_end_norm",   "mean"),
            mem_norm    =("mem_purchase_count_norm",  "mean"),
            wash_norm   =("ret_wash_count_norm",      "mean"),
            rev_norm    =("total_income_norm",        "mean"),
        )
        .dropna(subset=["new_sig_norm", "active_norm", "active_acct_norm"])
        .reset_index()
    )

    spike_agg["combined"] = (
        spike_agg["new_sig_norm"] + spike_agg["active_norm"] + spike_agg["active_acct_norm"]
    ) / 2
    threshold = spike_agg["combined"].median()
    spike_agg["bucket"] = spike_agg["combined"].apply(
        lambda v: "Customer-Facing" if v >= threshold else "Operational"
    )

    n_cf = (spike_agg["bucket"] == "Customer-Facing").sum()
    n_op = (spike_agg["bucket"] == "Operational").sum()

    promo_lb = promo_mm_df.merge(
        spike_agg[["site_key", "spike_date", "bucket", "combined"]],
        on=["site_key", "spike_date"], how="inner",
    )

    CF = promo_lb[promo_lb["bucket"] == "Customer-Facing"]
    OP = promo_lb[promo_lb["bucket"] == "Operational"]

    # ── Step 2: Event study helper ────────────────────────────────────────────
    MONTHS   = list(range(-6, 13))
    X_LABELS = [("M=0" if m == 0 else f"M{m:+d}") for m in MONTHS]

    def study(df, col):
        out = []
        for mfs, lbl in zip(MONTHS, X_LABELS):
            v = df.loc[df["months_from_spike"] == mfs, col].dropna()
            out.append({"lbl": lbl, "med": v.median(),
                        "q25": v.quantile(0.25), "q75": v.quantile(0.75)})
        return pd.DataFrame(out)

    PANELS = [
        ("new_signups_norm",        "New Sign-ups",         "classification signal"),
        ("active_members_norm",     "Active Members",       "classification signal"),
        ("mem_purchase_count_norm", "Membership Purchases", "independent"),
        ("ret_wash_count_norm",     "Retail Wash Count",    "independent ← moves opposite direction"),
    ]

    # ── Step 3: 4-row event study comparison ─────────────────────────────────
    GREEN  = "#1B5E20"; GREENBG = "rgba(27,94,32,0.12)"
    SLATE  = "#546E7A"; SLATEBG = "rgba(84,110,122,0.12)"

    fig1 = make_subplots(
        rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.08,
        subplot_titles=[f"{p[1]}  <i>({p[2]})</i>" for p in PANELS],
    )

    for r, (col, label, _) in enumerate(PANELS, 1):
        cf_s = study(CF, col)
        op_s = study(OP, col)

        for s, name, lc, bgc, dash in [
            (cf_s, f"Customer-Facing  (n={n_cf})", GREEN, GREENBG, "solid"),
            (op_s, f"Operational  (n={n_op})",     SLATE, SLATEBG, "dash"),
        ]:
            # IQR band
            fig1.add_trace(go.Scatter(
                x=list(s["lbl"]) + list(s["lbl"])[::-1],
                y=list(s["q75"]) + list(s["q25"])[::-1],
                fill="toself", fillcolor=bgc,
                line=dict(width=0), showlegend=False, hoverinfo="skip",
            ), row=r, col=1)
            # Median line
            fig1.add_trace(go.Scatter(
                x=s["lbl"], y=s["med"],
                name=name, showlegend=(r == 1),
                line=dict(color=lc, width=2.2, dash=dash),
                marker=dict(size=6, color=lc), mode="lines+markers",
                customdata=[[
                    f"<b>{label}</b> · {name}<br>"
                    f"Month: {lbl}<br>"
                    f"Median: {med:.3f}×<br>"
                    f"IQR: [{q25:.3f} – {q75:.3f}]"
                ] for lbl, med, q25, q75 in zip(s["lbl"], s["med"], s["q25"], s["q75"])],
                hovertemplate="%{customdata[0]}<extra></extra>",
            ), row=r, col=1)

        # Baseline 1.0 reference
        fig1.add_trace(go.Scatter(
            x=X_LABELS, y=[1.0] * len(X_LABELS),
            line=dict(color="rgba(0,0,0,0.18)", width=1, dash="dot"),
            mode="lines", showlegend=False, hoverinfo="skip",
        ), row=r, col=1)

    fig1.update_layout(
        title=dict(
            text=(
                "<b>OPEX Spike Classification: Customer-Facing vs Operational</b>"
                f"<br><sub>"
                f"Classification signal: (new_signups_norm + active_members_norm + accounts_end_norm) / 2 at M+1..M+3 "
                f"— median threshold = {threshold:.3f}  ·  "
                f"dotted line = pre-spike baseline (1.0×)  ·  "
                f"shaded bands = IQR  ·  M=0 = OPEX spike month"
                f"</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        legend=dict(orientation="h", y=-0.03, x=0.5, xanchor="center", font_size=9),
        height=1020,
        font_size=10,
        margin=dict(l=75, r=20, t=145, b=70),
        plot_bgcolor="rgba(250,250,250,1)",
        paper_bgcolor="white",
    )
    for r in range(1, 5):
        fig1.update_yaxes(
            title_text="× baseline", title_standoff=4,
            showgrid=True, gridcolor="rgba(200,200,200,0.4)",
            gridwidth=0.5, zeroline=False, row=r, col=1,
        )
        fig1.update_xaxes(showgrid=False, row=r, col=1)
    fig1.update_xaxes(title_text="Month relative to OPEX spike", row=4, col=1)

    # M=0 dotted vertical line on every panel
    for r_i in range(1, 5):
        yref_str = "y" if r_i == 1 else f"y{r_i}"
        fig1.add_shape(
            type="line",
            x0="M=0", x1="M=0", y0=0, y1=1,
            xref="x", yref=f"{yref_str} domain",
            line=dict(dash="dot", color="rgba(230,81,0,0.65)", width=1.5),
            layer="above",
        )

    fig1.show()

    # ── Step 4: Classification scatter ───────────────────────────────────────
    cf_pts = spike_agg[spike_agg["bucket"] == "Customer-Facing"]
    op_pts = spike_agg[spike_agg["bucket"] == "Operational"]

    fig2 = go.Figure()
    for pts, name, color, sym in [
        (cf_pts, f"Customer-Facing (n={n_cf})", GREEN, "circle"),
        (op_pts, f"Operational (n={n_op})",     SLATE, "x"),
    ]:
        fig2.add_trace(go.Scatter(
            x=pts["new_sig_norm"], y=pts["active_norm"],
            mode="markers", name=name,
            marker=dict(color=color, size=7, opacity=0.65, symbol=sym),
            customdata=list(zip(
                pts["site_key"], pts["spike_date"].dt.strftime("%Y-%m"),
                pts["new_sig_norm"].round(2), pts["active_norm"].round(2),
                pts["mem_norm"].round(2), pts["wash_norm"].round(2),
            )),
            hovertemplate=(
                "<b>%{customdata[0]}</b>  ·  %{customdata[1]}<br>"
                "New sign-ups: %{customdata[2]}×  ·  Active members: %{customdata[3]}×<br>"
                "Membership purchases: %{customdata[4]}×  ·  Wash count: %{customdata[5]}×"
                "<extra></extra>"
            ),
        ))

    # Median cross-hairs
    xm = spike_agg["new_sig_norm"].median()
    ym = spike_agg["active_norm"].median()
    fig2.add_vline(x=xm, line_dash="dot", line_color="rgba(0,0,0,0.22)", line_width=1.2)
    fig2.add_hline(y=ym, line_dash="dot", line_color="rgba(0,0,0,0.22)", line_width=1.2)

    # Quadrant labels
    xlim = spike_agg["new_sig_norm"].quantile(0.96)
    ylim = spike_agg["active_norm"].quantile(0.96)
    for txt, xq, yq, clr in [
        ("High sign-ups<br>High members",   xlim*0.82, ylim*0.92, GREEN),
        ("Low sign-ups<br>High members",    xm*0.30,   ylim*0.92, "#E65100"),
        ("High sign-ups<br>Low members",    xlim*0.82, ym*0.96,   "#1565C0"),
        ("Low sign-ups<br>Low members",     xm*0.30,   ym*0.96,   SLATE),
    ]:
        fig2.add_annotation(x=xq, y=yq, text=f"<i>{txt}</i>",
                            showarrow=False, font=dict(size=8, color=clr),
                            bgcolor="rgba(255,255,255,0.65)")

    fig2.update_layout(
        title=dict(
            text=(
                "<b>Each Dot = One OPEX Spike Event</b>"
                f"<br><sub>X = avg new sign-ups M+1..M+3 (normalised)  ·  "
                "Y = avg active members M+1..M+3 (normalised)  ·  "
                "dashed lines = median thresholds used for classification  ·  "
                "hover for site & date detail</sub>"
            ),
            x=0.02, xanchor="left",
        ),
        xaxis_title="New Sign-ups at M+1–M+3  (× pre-spike baseline)",
        yaxis_title="Active Members at M+1–M+3  (× pre-spike baseline)",
        height=580, font_size=10,
        margin=dict(l=75, r=20, t=130, b=60),
        plot_bgcolor="rgba(250,250,250,1)", paper_bgcolor="white",
        legend=dict(orientation="h", y=-0.1, x=0.5, xanchor="center"),
    )
    fig2.update_xaxes(showgrid=True, gridcolor="rgba(200,200,200,0.4)", zeroline=False)
    fig2.update_yaxes(showgrid=True, gridcolor="rgba(200,200,200,0.4)", zeroline=False)
    fig2.show()

    # ── Step 5: Top spikes per bucket ─────────────────────────────────────────
    COLS = ["site_key","spike_date","new_sig_norm","active_norm","active_acct_norm","mem_norm","wash_norm","combined"]
    print(f"\n{'─'*72}")
    print(f" TOP 10 CUSTOMER-FACING SPIKES  (combined score desc)")
    print(f"{'─'*72}")
    print(spike_agg[spike_agg["bucket"]=="Customer-Facing"]
          .nlargest(10,"combined")[COLS].round(3).to_string(index=False))

    print(f"\n{'─'*72}")
    print(f" TOP 10 OPERATIONAL SPIKES  (combined score asc = most purely operational)")
    print(f"{'─'*72}")
    print(spike_agg[spike_agg["bucket"]=="Operational"]
          .nsmallest(10,"combined")[COLS].round(3).to_string(index=False))

    # Store for downstream use
    spike_buckets = spike_agg.copy()
    print(f"\nspike_buckets saved ({len(spike_buckets)} rows) — columns: {list(spike_buckets.columns)}")



────────────────────────────────────────────────────────────────────────
 TOP 10 CUSTOMER-FACING SPIKES  (combined score desc)
────────────────────────────────────────────────────────────────────────
               site_key spike_date  new_sig_norm  active_norm  active_acct_norm  mem_norm  wash_norm  combined
southernshine_000690__3 2025-08-31         3.299        7.115             7.193     9.803      1.990     8.803
southernshine_000690__3 2025-09-30         2.667        5.779             5.818     8.170      0.958     7.132
southernshine_000690__3 2025-10-31         1.585        4.601             4.622     6.601      0.601     5.404
     peppyscw_000491__4 2026-03-31         1.669        4.504             4.550     4.458      1.820     5.361
     peppyscw_000491__4 2026-04-30         1.393        3.199             3.222     3.225      1.305     3.907
  clearwater_000397__16 2024-08-31         1.040        2.592             2.583     2.751      1.438     3.107
 happycarwash_000394__